# 🚀 V8.1 Directive R3: Host-Safe Trim Submission (Dual T4)
### Solar Filament Segmentation Challenge 2026 (IEEE BigData Cup)
**Authority**: Grok Master | **Executor**: Antigravity  
**Baseline Anchor**: 0.350 Public LB (1,231 rows across 178 disks)  
**Cascade**: YOLO11s-seg @ 1024 -> Adaptive Square Crop -> Crop U-Net (ResNet-34 @ 384) with 4-flip TTA -> Greedy Pixel-Carve Sanitizer -> COCO RLE

---
### 📋 Execution Guide (Fast Path: ~4–8 minutes):
1. **Inputs Attached**:
   - Competition data: `filament-segmentation-2026`
   - Previous run: `notebooka464bcfe13` (via **+ Add Input -> Your Work**)
2. **Frozen R3 Submit Knobs**:
   - `conf = 0.20`
   - `min_area = 400`
   - `yolo_fallback = 0`
   - `overlap_mode = trim` (host-safe: zero shared pixels guaranteed)
   - `tta = True` (4-flip TTA on crops)
   - Same weights as `notebooka464bcfe13` (`best.pt` & `crop_refiner_r34.pth`)
3. **No Retrain / No 80-Point Sweep**: Stage C test inference only (~3–5 min).
4. **Mandatory Host Sanitizer & Audit Assertion**:
   - Every disk strictly asserted for `sum(mask_i & mask_j) == 0` for all $i 
e j$.
   - Fails the notebook if even a single shared pixel exists.


In [ ]:
# ==============================================================================
# CELL 2: Environment Setup & Pinned Dependency Freeze
# Purpose: Configure PyTorch memory allocator, install frozen production packages,
#          and log runtime package versions for complete reproducibility.
# Runtime: ~30 seconds
# ==============================================================================
import os, sys, time
os.environ["PYTHONUNBUFFERED"] = "1"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

t_notebook_start = time.perf_counter()

# Install pinned production dependencies (Internet ON)
!pip -q install ultralytics==8.4.103 segmentation-models-pytorch pycocotools albumentations timm

# Safe version resolution using importlib.metadata (cannot crash if package lacks __version__)
from importlib.metadata import version as pkg_version
import sys, torch, cv2, numpy as np, pandas as pd

def safe_version(pkg_name):
    try:
        return pkg_version(pkg_name)
    except Exception:
        return "unknown"

print("=" * 75)
print("DEPENDENCY REPRODUCIBILITY FREEZE")
print("=" * 75)
print(f"  Python:                       {sys.version.split()[0]}")
print(f"  PyTorch:                      {torch.__version__}")
print(f"  Ultralytics:                  {safe_version('ultralytics')}")
print(f"  segmentation_models_pytorch:  {safe_version('segmentation-models-pytorch')}")
print(f"  pycocotools:                  {safe_version('pycocotools')}")
print(f"  albumentations:               {safe_version('albumentations')}")
print(f"  timm:                         {safe_version('timm')}")
print(f"  OpenCV (cv2):                 {cv2.__version__}")
print(f"  NumPy:                        {np.__version__}")
print(f"  Pandas:                       {pd.__version__}")
print("=" * 75)


In [ ]:
# ==============================================================================
# CELL 3: Hardware Diagnostics & Competition Dataset Detection
# Purpose: Verify Dual T4 GPUs, inspect available VRAM, and locate MAGFiLO dataset.
# Runtime: ~2 seconds
# ==============================================================================
import torch
from pathlib import Path

print("=" * 75)
print("SYSTEM & ACCELERATOR DIAGNOSTIC")
print("=" * 75)
n_gpus = torch.cuda.device_count()
print(f"CUDA Available: {torch.cuda.is_available()} | GPU Count: {n_gpus}")

for i in range(n_gpus):
    props = torch.cuda.get_device_properties(i)
    vram_gb = props.total_memory / (1024 ** 3)
    print(f"  [GPU {i}] {props.name} | VRAM: {vram_gb:.2f} GB")

assert torch.cuda.is_available(), "FATAL: GPU is required! Switch runtime to GPU T4 x2."

# Detect competition dataset paths in Kaggle input
candidate_paths = [
    Path("/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026"),
    Path("/kaggle/input/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026"),
    Path("/kaggle/input/filament-segmentation-2026"),
    Path("data/MAGFiLO_1.0_Kaggle_2026"),
]
data_dir = next((p for p in candidate_paths if p.exists()), None)
print(f"Dataset Root: {data_dir}")
assert data_dir is not None, "FATAL: Dataset not found in /kaggle/input! Attach filament-segmentation-2026."
print("=" * 75)


In [ ]:
# ==============================================================================
# CELL 4: Modular Code Deployment
# Purpose: Unpack modular production scripts into /kaggle/working directory:
#          - metrics/pq.py: Official Kirillov Panoptic Quality implementation
#          - scripts/convert_coco_to_yolo.py: COCO to YOLO-seg converter with GroupKFold
#          - v8_1/config.py: Frozen hyperparameters (YOLO batch 2, Refiner batch 8)
#          - v8_1/geometry.py: Boundary-safe square crop and resize math
#          - v8_1/1_train_yolo.py: YOLO11s-seg training & multi-annotator PQ sweep
#          - v8_1/2_train_crop_refiner.py: ResNet-34 Crop U-Net refiner training
#          - v8_1/3_infer_cascade.py: Dual-T4 model-parallel cascade inference engine
#          - v8_1/4_sweep_inference.py: Grok Directive R2 diagnostic & grid sweep engine
# Runtime: ~1 second
# ==============================================================================
from pathlib import Path

# Create module directories in /kaggle/working
Path("metrics").mkdir(parents=True, exist_ok=True)
Path("scripts").mkdir(parents=True, exist_ok=True)
Path("v8_1").mkdir(parents=True, exist_ok=True)

with open("metrics/__init__.py", "w") as f: f.write("")
with open("v8_1/__init__.py", "w") as f: f.write("")

with open("metrics/pq.py", "w", encoding="utf-8") as f:
    f.write('"""\nKirillov Panoptic Quality (PQ) scorer for solar filament instance segmentation.\n\nPQ = SQ × RQ\nSQ = mean IoU of matched instances (Segmentation Quality)\nRQ = TP / (TP + 0.5·FP + 0.5·FN) (Recognition Quality)\n\nMatching: greedy unique pairing with IoU > 0.5 threshold.\nReference: Kirillov et al., "Panoptic Segmentation", CVPR 2019.\n"""\n\nimport numpy as np\nimport pycocotools.mask as mask_utils\n\n\ndef encode_mask(mask_hw: np.ndarray) -> str:\n    """Encode a 2D binary mask (H, W) to a COCO RLE counts string.\n\n    Always goes through pycocotools Fortran 3D encode.\n    Never returns a hardcoded string.\n\n    Args:\n        mask_hw: uint8 array of shape (H, W) with values 0 or 1.\n\n    Returns:\n        COCO RLE counts string (e.g. for a 2048×2048 zero mask this\n        will be whatever pycocotools produces — currently \'PPPP4\').\n    """\n    h, w = mask_hw.shape[:2]\n    mask_3d = np.asfortranarray(mask_hw.astype(np.uint8)).reshape((h, w, 1))\n    rle = mask_utils.encode(mask_3d)[0]\n    counts = rle["counts"]\n    if isinstance(counts, bytes):\n        counts = counts.decode("utf-8")\n    return counts\n\n\ndef decode_rle(counts_str: str, h: int | tuple = 2048, w: int = 2048) -> np.ndarray:\n    """Decode a COCO RLE counts string back to a 2D binary mask (H, W)."""\n    if isinstance(h, (tuple, list)):\n        h, w = h[0], h[1]\n    rle = {"size": [int(h), int(w)], "counts": counts_str}\n    mask = mask_utils.decode(rle)\n    if mask.ndim == 3:\n        mask = mask[:, :, 0]\n    return mask\n\n\ndef _iou(mask_a: np.ndarray, mask_b: np.ndarray) -> float:\n    """Compute IoU between two binary masks."""\n    intersection = np.logical_and(mask_a, mask_b).sum()\n    union = np.logical_or(mask_a, mask_b).sum()\n    if union == 0:\n        return 0.0\n    return float(intersection) / float(union)\n\n\ndef pq_score(\n    pred_masks: list[np.ndarray],\n    gt_masks: list[np.ndarray],\n    iou_threshold: float = 0.5,\n) -> dict:\n    """Compute Kirillov Panoptic Quality between predicted and GT instances.\n\n    Matching is greedy: sort all (pred, gt) pairs by descending IoU,\n    accept a pair only if IoU > iou_threshold and neither instance is\n    already matched. Each instance can match at most once.\n\n    Args:\n        pred_masks: list of binary (H, W) uint8 arrays, one per predicted instance.\n        gt_masks:   list of binary (H, W) uint8 arrays, one per GT instance.\n        iou_threshold: minimum IoU for a valid match (default 0.5).\n\n    Returns:\n        dict with keys: PQ, SQ, RQ, TP, FP, FN, matched_ious.\n    """\n    n_pred = len(pred_masks)\n    n_gt = len(gt_masks)\n\n    # Edge cases\n    if n_pred == 0 and n_gt == 0:\n        return {"PQ": 1.0, "SQ": 1.0, "RQ": 1.0, "TP": 0, "FP": 0, "FN": 0, "matched_ious": []}\n    if n_pred == 0:\n        return {"PQ": 0.0, "SQ": 0.0, "RQ": 0.0, "TP": 0, "FP": 0, "FN": n_gt, "matched_ious": []}\n    if n_gt == 0:\n        return {"PQ": 0.0, "SQ": 0.0, "RQ": 0.0, "TP": 0, "FP": n_pred, "FN": 0, "matched_ious": []}\n\n    # Compute full IoU matrix\n    iou_matrix = np.zeros((n_pred, n_gt), dtype=np.float64)\n    for i, pm in enumerate(pred_masks):\n        for j, gm in enumerate(gt_masks):\n            iou_matrix[i, j] = _iou(pm, gm)\n\n    # Greedy matching: sort all pairs by descending IoU\n    pairs = []\n    for i in range(n_pred):\n        for j in range(n_gt):\n            if iou_matrix[i, j] > iou_threshold:\n                pairs.append((iou_matrix[i, j], i, j))\n    pairs.sort(key=lambda x: x[0], reverse=True)\n\n    matched_pred = set()\n    matched_gt = set()\n    matched_ious = []\n\n    for iou_val, pi, gi in pairs:\n        if pi in matched_pred or gi in matched_gt:\n            continue\n        matched_pred.add(pi)\n        matched_gt.add(gi)\n        matched_ious.append(iou_val)\n\n    tp = len(matched_ious)\n    fp = n_pred - tp\n    fn = n_gt - tp\n\n    sq = float(np.mean(matched_ious)) if tp > 0 else 0.0\n    rq = tp / (tp + 0.5 * fp + 0.5 * fn) if (tp + fp + fn) > 0 else 0.0\n    pq = sq * rq\n\n    return {\n        "PQ": round(pq, 6),\n        "SQ": round(sq, 6),\n        "RQ": round(rq, 6),\n        "TP": tp,\n        "FP": fp,\n        "FN": fn,\n        "matched_ious": matched_ious,\n    }\n\n\ndef pq_score_multi(\n    pred_masks_list: list[list[np.ndarray]],\n    gt_masks_list: list[list[np.ndarray]],\n    iou_threshold: float = 0.5,\n) -> dict:\n    """Compute mean PQ / SQ / RQ across multiple images.\n\n    Args:\n        pred_masks_list: list of per-image predicted mask lists.\n        gt_masks_list:   list of per-image GT mask lists.\n        iou_threshold: minimum IoU for matching.\n\n    Returns:\n        dict with mean PQ, SQ, RQ, total TP, FP, FN, and per-image results.\n    """\n    assert len(pred_masks_list) == len(gt_masks_list), "Mismatched image count"\n\n    per_image = []\n    total_tp = total_fp = total_fn = 0\n    sum_pq = sum_sq = sum_rq = 0.0\n\n    for preds, gts in zip(pred_masks_list, gt_masks_list):\n        result = pq_score(preds, gts, iou_threshold)\n        per_image.append(result)\n        total_tp += result["TP"]\n        total_fp += result["FP"]\n        total_fn += result["FN"]\n        sum_pq += result["PQ"]\n        sum_sq += result["SQ"]\n        sum_rq += result["RQ"]\n\n    n = len(pred_masks_list)\n    return {\n        "mean_PQ": round(sum_pq / n, 6) if n > 0 else 0.0,\n        "mean_SQ": round(sum_sq / n, 6) if n > 0 else 0.0,\n        "mean_RQ": round(sum_rq / n, 6) if n > 0 else 0.0,\n        "total_TP": total_tp,\n        "total_FP": total_fp,\n        "total_FN": total_fn,\n        "n_images": n,\n        "per_image": per_image,\n    }\n')

with open("scripts/convert_coco_to_yolo.py", "w", encoding="utf-8") as f:
    f.write('"""\nConvert COCO Annotations to YOLO-seg format for MAGFiLO Dataset.\n\nRequirements (Grok Phase B Verdict):\n1. Key off JSON image_id, not pooled filename (one annotator observation = one YOLO sample).\n2. No polygon OR-merging (eliminates 11.6 polygon/image label inflation bug).\n3. Include categories 1, 2, 3, 4 as solar filaments.\n4. Single class \'filament\' (id 0) for YOLO-seg.\n5. Verification and reporting:\n   - JSON images count\n   - JPEGs on disk count\n   - file_name missing on disk\n   - JPEGs with zero annotations\n   - polygons skipped (too few points, zero/degenerate area)\n6. Exit non-zero if > 5% of JSON file_names are missing unless --allow-missing.\n7. GroupKFold by year prefix to avoid multi-annotator and temporal leakage.\n8. Write data/yolo_seg/data.yaml and conversion_report.json.\n"""\n\nimport argparse\nimport json\nimport os\nimport shutil\nimport sys\nfrom collections import defaultdict\nfrom pathlib import Path\n\nimport numpy as np\nimport pandas as pd\nfrom sklearn.model_selection import GroupKFold\n\n# Ensure project root is on sys.path\nPROJECT_ROOT = Path(__file__).resolve().parent.parent\nsys.path.insert(0, str(PROJECT_ROOT))\n\n\ndef polygon_area(pts: np.ndarray) -> float:\n    """Compute polygon area using Shoelace formula. pts: (N, 2)."""\n    if len(pts) < 3:\n        return 0.0\n    x = pts[:, 0]\n    y = pts[:, 1]\n    return 0.5 * float(np.abs(np.dot(x, np.roll(y, 1)) - np.dot(y, np.roll(x, 1))))\n\n\ndef materialize_file(src: Path, dst: Path, mode: str = "auto") -> str:\n    """Safely materialize src at dst.\n    Preferred order:\n      1. symlink (robust cross-mount support on Linux/Kaggle without copying 700MB)\n      2. hardlink (same filesystem zero-copy)\n      3. shutil.copy2 (universal fallback)\n    Verifies destination exists and is non-empty.\n    """\n    if dst.exists():\n        try:\n            dst.unlink()\n        except Exception:\n            pass\n\n    if mode == "symlink":\n        candidates = ["symlink", "hardlink", "copy2"]\n    elif mode == "hardlink":\n        candidates = ["hardlink", "symlink", "copy2"]\n    elif mode == "copy":\n        candidates = ["copy2"]\n    else:  # "auto"\n        candidates = ["symlink", "hardlink", "copy2"]\n\n    for cand in candidates:\n        try:\n            if cand == "symlink":\n                os.symlink(src.resolve(), dst)\n            elif cand == "hardlink":\n                os.link(src, dst)\n            elif cand == "copy2":\n                shutil.copy2(src, dst)\n\n            if dst.exists() and dst.stat().st_size > 0:\n                return cand\n            else:\n                if dst.exists():\n                    dst.unlink()\n        except Exception:\n            if dst.exists():\n                try:\n                    dst.unlink()\n                except Exception:\n                    pass\n            continue\n\n    raise RuntimeError(f"FATAL: Failed to materialize {src} to {dst} with any method.")\n\n\ndef convert_coco_to_yolo(\n    data_dir: Path,\n    out_dir: Path,\n    val_fold: int = 0,\n    n_splits: int = 5,\n    allow_missing: bool = False,\n    link_mode: str = "auto",  # \'hardlink\', \'symlink\', \'copy\'\n) -> dict:\n    """Perform annotator-aware COCO to YOLO-seg conversion."""\n\n    train_dir = data_dir / "train"\n    img_dir = train_dir / "train_images"\n    json_candidates = list(train_dir.glob("*.json")) + list(data_dir.glob("*.json"))\n    if not json_candidates:\n        raise FileNotFoundError(f"No annotation JSON found in {data_dir}")\n    json_path = json_candidates[0]\n\n    print("=" * 70)\n    print("COCO -> YOLO-SEG CONVERTER (Annotator-Aware, Zero-OR)")\n    print("=" * 70)\n    print(f"  Annotation JSON: {json_path}")\n    print(f"  Image directory: {img_dir}")\n    print(f"  Output root:     {out_dir}")\n    print(f"  Val fold:        {val_fold} (of {n_splits})")\n    print()\n\n    with open(json_path, "r", encoding="utf-8") as f:\n        coco = json.load(f)\n\n    json_images = coco.get("images", [])\n    annotations = coco.get("annotations", [])\n    categories = coco.get("categories", [])\n\n    print("PRE-FLIGHT DATASET AUDIT:")\n    print(f"  1. JSON images count:          {len(json_images)}")\n    \n    # Check JPEGs on disk\n    jpegs_on_disk = list(img_dir.glob("*.jpeg")) + list(img_dir.glob("*.jpg"))\n    disk_filenames = {p.name for p in jpegs_on_disk}\n    print(f"  2. JPEGs on disk:              {len(jpegs_on_disk)}")\n\n    # Check missing file_names\n    json_file_names = {img["file_name"] for img in json_images}\n    missing_files = sorted(list(json_file_names - disk_filenames))\n    pct_missing = (len(missing_files) / len(json_file_names) * 100.0) if json_file_names else 0.0\n    print(f"  3. file_name missing on disk:  {len(missing_files)} ({pct_missing:.2f}%)")\n    if missing_files:\n        print(f"     Sample missing: {missing_files[:5]}")\n\n    if pct_missing > 5.0 and not allow_missing:\n        print(f"❌ ERROR: More than 5% of JSON file_names are missing ({pct_missing:.2f}%). Exiting.")\n        sys.exit(1)\n\n    # Annotations per image_id and per physical file_name\n    img_id_to_anns = defaultdict(list)\n    fn_to_anns = defaultdict(list)\n    img_id_to_record = {img["id"]: img for img in json_images}\n\n    for ann in annotations:\n        img_id = ann["image_id"]\n        img_id_to_anns[img_id].append(ann)\n        if img_id in img_id_to_record:\n            fn_to_anns[img_id_to_record[img_id]["file_name"]].append(ann)\n\n    zero_ann_jpegs = [fn for fn in disk_filenames if len(fn_to_anns[fn]) == 0]\n    print(f"  4. JPEGs with zero annotations: {len(zero_ann_jpegs)}")\n\n    # Category distribution audit\n    from collections import Counter\n    cat_counts = Counter(ann.get("category_id") for ann in annotations)\n    cat_1_count = cat_counts.get(1, 0)\n    cat_2_count = cat_counts.get(2, 0)\n    cat_3_count = cat_counts.get(3, 0)\n    cat_4_count = cat_counts.get(4, 0)\n\n    # Polygon validation audit (All categories 1, 2, 3, 4 are valid filaments for class 0)\n    skipped_points = 0\n    skipped_area = 0\n    valid_polys_count = 0\n\n    valid_anns_by_img_id = defaultdict(list)\n\n    for ann in annotations:\n        # All categories are mapped to class 0 \'filament\'\n        img_id = ann["image_id"]\n        if img_id not in img_id_to_record:\n            continue\n        \n        img_rec = img_id_to_record[img_id]\n        h = float(img_rec.get("height", 2048))\n        w = float(img_rec.get("width", 2048))\n\n        segs = ann.get("segmentation", [])\n        if not isinstance(segs, list):\n            continue\n\n        for poly in segs:\n            if not isinstance(poly, list) or len(poly) < 6 or len(poly) % 2 != 0:\n                skipped_points += 1\n                continue\n\n            pts = np.asarray(poly, dtype=np.float32).reshape(-1, 2)\n            if np.any(np.isnan(pts)) or np.any(np.isinf(pts)):\n                skipped_points += 1\n                continue\n\n            area = polygon_area(pts)\n            if area <= 0.0:\n                skipped_area += 1\n                continue\n\n            # Normalized coordinates for YOLO: [0, 1]\n            norm_pts = pts.copy()\n            norm_pts[:, 0] = np.clip(norm_pts[:, 0] / w, 0.0, 1.0)\n            norm_pts[:, 1] = np.clip(norm_pts[:, 1] / h, 0.0, 1.0)\n\n            valid_polys_count += 1\n            valid_anns_by_img_id[img_id].append(norm_pts.reshape(-1).tolist())\n\n    print(f"  5. Categories & Polygons audit (All categories -> Class 0 \'filament\'):")\n    print(f"     - Category 1 (Left):           {cat_1_count}")\n    print(f"     - Category 2 (Right):          {cat_2_count}")\n    print(f"     - Category 3 (Unidentifiable): {cat_3_count}")\n    print(f"     - Category 4 (Ambiguous):      {cat_4_count}")\n    print(f"     - Valid filaments:             {valid_polys_count}")\n    print(f"     - Skipped (<6 pts/coords):     {skipped_points}")\n    print(f"     - Skipped (zero area):         {skipped_area}")\n    print()\n\n    # Determine train/val split using GroupKFold by year\n    # Each sample corresponds to a single JSON image_id\n    sample_records = []\n    for img_rec in json_images:\n        fn = img_rec["file_name"]\n        if fn not in disk_filenames:\n            continue\n        img_id = img_rec["id"]\n        # Group by year prefix (first 4 digits of filename)\n        year_group = fn[:4] if fn[:4].isdigit() else "0000"\n        sample_records.append({\n            "image_id": img_id,\n            "file_name": fn,\n            "group": year_group,\n            "n_polys": len(valid_anns_by_img_id[img_id]),\n        })\n\n    df_samples = pd.DataFrame(sample_records)\n    gkf = GroupKFold(n_splits=n_splits)\n    df_samples["fold"] = -1\n    for fold_idx, (trn_idx, v_idx) in enumerate(gkf.split(df_samples, groups=df_samples["group"])):\n        df_samples.loc[v_idx, "fold"] = fold_idx\n\n    val_mask = df_samples["fold"] == val_fold\n    train_samples = df_samples[~val_mask]\n    val_samples = df_samples[val_mask]\n\n    print(f"SPLIT ASSIGNMENT (GroupKFold by year, val_fold={val_fold}):")\n    print(f"  Train samples (annotator observations): {len(train_samples)}")\n    print(f"  Val samples (annotator observations):   {len(val_samples)}")\n    print(f"  Train unique physical JPEGs:            {train_samples[\'file_name\'].nunique()}")\n    print(f"  Val unique physical JPEGs:              {val_samples[\'file_name\'].nunique()}")\n    print()\n\n    # Setup directories\n    if out_dir.exists():\n        shutil.rmtree(out_dir)\n\n    for split in ["train", "val"]:\n        (out_dir / "images" / split).mkdir(parents=True, exist_ok=True)\n        (out_dir / "labels" / split).mkdir(parents=True, exist_ok=True)\n\n    # Write samples\n    print("WRITING SAMPLES TO DISK...")\n    \n    # Test link / materialization strategy\n    test_src = img_dir / sample_records[0]["file_name"]\n    test_dst = out_dir / "_test_materialize"\n    detected_method = materialize_file(test_src, test_dst, mode=link_mode)\n    if test_dst.exists():\n        test_dst.unlink()\n\n    print(f"  File materialization method verified: {detected_method}")\n\n    written_counts = {"train": 0, "val": 0}\n\n    for idx, row in df_samples.iterrows():\n        split = "val" if row["fold"] == val_fold else "train"\n        img_id = row["image_id"]\n        fn = row["file_name"]\n        stem = Path(fn).stem\n        # Distinct sample name keyed off image_id\n        sample_stem = f"{stem}_ann_{img_id}"\n\n        src_img = img_dir / fn\n        dst_img = out_dir / "images" / split / f"{sample_stem}.jpeg"\n        dst_lbl = out_dir / "labels" / split / f"{sample_stem}.txt"\n\n        # Safely materialize image with existence & non-zero size verification\n        materialize_file(src_img, dst_img, mode=detected_method)\n        if not (dst_img.exists() and dst_img.stat().st_size > 0):\n            raise RuntimeError(f"FATAL: Materialized image {dst_img} is missing or 0 bytes!")\n\n        # Write YOLO segmentation label file\n        polys = valid_anns_by_img_id[img_id]\n        with open(dst_lbl, "w", encoding="utf-8") as lf:\n            for p in polys:\n                coord_str = " ".join(f"{c:.6f}" for c in p)\n                lf.write(f"0 {coord_str}\\n")\n\n        written_counts[split] += 1\n\n    print(f"  Wrote {written_counts[\'train\']} train files, {written_counts[\'val\']} val files")\n    print()\n\n    # Write data.yaml\n    yaml_content = f"""# YOLO-seg dataset config for Solar Filament Segmentation 2026\npath: {out_dir.resolve().as_posix()}\ntrain: images/train\nval: images/val\n\nnames:\n  0: filament\n"""\n    yaml_path = out_dir / "data.yaml"\n    with open(yaml_path, "w", encoding="utf-8") as f:\n        f.write(yaml_content)\n    print(f"  Written data.yaml: {yaml_path}")\n\n    # Write conversion_report.json\n    report = {\n        "json_path": str(json_path),\n        "json_images_count": len(json_images),\n        "jpegs_on_disk_count": len(jpegs_on_disk),\n        "unique_json_filenames": len(json_file_names),\n        "missing_on_disk_count": len(missing_files),\n        "missing_on_disk_pct": pct_missing,\n        "zero_ann_jpegs_count": len(zero_ann_jpegs),\n        "category_1_count": cat_1_count,\n        "category_2_count": cat_2_count,\n        "category_3_count": cat_3_count,\n        "category_4_count": cat_4_count,\n        "skipped_polygons_invalid_pts": skipped_points,\n        "skipped_polygons_zero_area": skipped_area,\n        "valid_polygons_count": valid_polys_count,\n        "train_samples_count": len(train_samples),\n        "val_samples_count": len(val_samples),\n        "train_unique_jpegs": int(train_samples["file_name"].nunique()),\n        "val_unique_jpegs": int(val_samples["file_name"].nunique()),\n        "val_fold": val_fold,\n        "n_splits": n_splits,\n        "linking_method": detected_method,\n        "classes": {"0": "filament"},\n    }\n    report_path = out_dir / "conversion_report.json"\n    with open(report_path, "w", encoding="utf-8") as f:\n        json.dump(report, f, indent=2)\n    print(f"  Written conversion_report.json: {report_path}")\n    print("=" * 70)\n\n    return report\n\n\ndef main():\n    parser = argparse.ArgumentParser(description="COCO to YOLO-seg dataset converter")\n    parser.add_argument(\n        "--data-dir",\n        type=str,\n        default="data/MAGFiLO_1.0_Kaggle_2026",\n        help="Path to MAGFiLO competition root",\n    )\n    parser.add_argument(\n        "--out-dir",\n        type=str,\n        default="data/yolo_seg",\n        help="Path to output YOLO dataset",\n    )\n    parser.add_argument("--val-fold", type=int, default=0, help="Fold index to use as val")\n    parser.add_argument("--n-splits", type=int, default=5, help="Number of GroupKFold splits")\n    parser.add_argument(\n        "--allow-missing",\n        action="store_true",\n        help="Allow missing images above 5% threshold without error",\n    )\n    parser.add_argument(\n        "--link-mode",\n        type=str,\n        default="auto",\n        choices=["auto", "hardlink", "symlink", "copy"],\n        help="How to place image files in YOLO folders",\n    )\n    args = parser.parse_args()\n\n    data_dir = PROJECT_ROOT / args.data_dir\n    out_dir = PROJECT_ROOT / args.out_dir\n\n    convert_coco_to_yolo(\n        data_dir=data_dir,\n        out_dir=out_dir,\n        val_fold=args.val_fold,\n        n_splits=args.n_splits,\n        allow_missing=args.allow_missing,\n        link_mode=args.link_mode,\n    )\n\n\nif __name__ == "__main__":\n    main()\n')

with open("v8_1/config.py", "w", encoding="utf-8") as f:
    f.write('"""\nv8_1 Configuration — Unified Frozen Hyperparameters for Seed -> Refine Cascade.\n\nMaster: ChatGPT | Builder: Antigravity\nTarget: Kaggle Solar Filament Segmentation Challenge 2026\n"""\n\nfrom pathlib import Path\nimport os\nimport torch\n\n# Base directories & environment detection\nKAGGLE = Path("/kaggle/input").exists()\nPROJECT_ROOT = Path(__file__).resolve().parent.parent\nOUT = Path("/kaggle/working") if KAGGLE else PROJECT_ROOT\n\nCANDIDATE_DATA_DIRS = [\n    Path("/kaggle/input/competitions/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026"),\n    Path("/kaggle/input/filament-segmentation-2026/MAGFiLO_1.0_Kaggle_2026"),\n    Path("/kaggle/input/filament-segmentation-2026"),\n    PROJECT_ROOT / "data" / "MAGFiLO_1.0_Kaggle_2026",\n]\n\nMAGFILO_DIR = next((p for p in CANDIDATE_DATA_DIRS if p.exists()), PROJECT_ROOT / "data" / "MAGFiLO_1.0_Kaggle_2026")\nYOLO_DATA_DIR = OUT / "data" / "yolo_seg"\nRUNS_DIR = OUT / "runs"\nMODELS_DIR = OUT / "models" / "v8_1"\nSUBMISSIONS_DIR = OUT if KAGGLE else (PROJECT_ROOT / "submissions")\n\n# Ensure output directories exist\nMODELS_DIR.mkdir(parents=True, exist_ok=True)\nSUBMISSIONS_DIR.mkdir(parents=True, exist_ok=True)\n\n\nclass YOLOConfig:\n    """Frozen YOLO11s-seg Stage 1 Proposer Specification."""\n    MODEL = "yolo11s-seg.pt"\n    IMGSZ = 1024\n    EPOCHS = 20\n    BATCH = 2  # Safe batch size on T4 at 1024x1024\n    DEVICE = 0  # Single GPU for training to prevent notebook subprocess DDP hang\n    AMP = True\n    MOSAIC = 0.5\n    CLOSE_MOSAIC = 2\n    DEGREES = 10.0\n    FLIPUD = 0.5\n    FLIPLR = 0.5\n    OVERLAP_MASK = True  # Pinned Ultralytics default for segmentation mask loss\n    MASK_RATIO = 4       # Pinned Ultralytics default downsample ratio\n    WORKERS = 2 if os.name != "nt" else 0  # 2 on Linux/Kaggle, 0 on Windows\n    PROJECT = str(RUNS_DIR)\n    NAME = "v8_1_yolo_s1024"\n    DATA_YAML = str(YOLO_DATA_DIR / "data.yaml")\n    WEIGHTS_PATH = RUNS_DIR / "v8_1_yolo_s1024" / "weights" / "best.pt"\n\n    @classmethod\n    def resolve_weights(cls, explicit_path=None) -> Path:\n        """Dynamically resolve best.pt path across input datasets and runs."""\n        if explicit_path and Path(explicit_path).exists():\n            return Path(explicit_path)\n        if cls.WEIGHTS_PATH.exists():\n            return cls.WEIGHTS_PATH\n        # Search /kaggle/input for any best.pt\n        if Path("/kaggle/input").exists():\n            input_candidates = sorted(list(Path("/kaggle/input").glob("**/best.pt")), key=lambda p: p.stat().st_mtime)\n            if input_candidates:\n                return input_candidates[-1]\n        # Search RUNS_DIR for any best.pt\n        candidates = sorted(list(RUNS_DIR.glob("**/weights/best.pt")), key=lambda p: p.stat().st_mtime)\n        if candidates:\n            return candidates[-1]\n        return cls.WEIGHTS_PATH\n\n\nclass CropRefinerConfig:\n    """Frozen Stage 2 Crop U-Net Specification."""\n    ENCODER = "resnet34"\n    IN_CHANNELS = 3  # [Raw, CLAHE, High-Pass Unsharp]\n    CLASSES = 1\n    CROP_MIN = 256\n    CROP_MAX = 512\n    CROP_PADDING = 1.2\n    BATCH_SIZE = 8\n    TARGET_SIZE = 384  # Square resized input size for refiner batch\n    EPOCHS = 20\n    LR = 5e-4\n    WEIGHT_DECAY = 1e-4\n    AMP = True\n    CKPT_PATH = MODELS_DIR / "crop_refiner_r34.pth"\n\n    @classmethod\n    def resolve_ckpt(cls, explicit_path=None) -> Path:\n        """Dynamically resolve crop_refiner_r34.pth across input datasets and models."""\n        if explicit_path and Path(explicit_path).exists():\n            return Path(explicit_path)\n        if cls.CKPT_PATH.exists():\n            return cls.CKPT_PATH\n        if Path("/kaggle/input").exists():\n            input_candidates = sorted(list(Path("/kaggle/input").glob("**/crop_refiner_r34.pth")), key=lambda p: p.stat().st_mtime)\n            if input_candidates:\n                return input_candidates[-1]\n        candidates = sorted(list(MODELS_DIR.glob("**/crop_refiner_r34.pth")), key=lambda p: p.stat().st_mtime)\n        if candidates:\n            return candidates[-1]\n        return cls.CKPT_PATH\n\n\nclass CascadeConfig:\n    """Frozen Stage 3 Instance Cascade Inference Specification (Directive R3)."""\n    RESIDUAL_DEFAULT = 0  # Off by default for Baseline 1\n    RESIDUAL_IOU_THRESH = 0.30  # Keep residual only if IoU < 0.30 vs all YOLO instances\n    DEFAULT_CONF = 0.20\n    DEFAULT_MIN_AREA = 400\n    DEFAULT_OVERLAP_MODE = "trim"\n    DEFAULT_FALLBACK = 0\n    SOLAR_DISK_R_FRAC = 0.93  # Geometric radius clipping\n    TTA_4FLIP = True  # 4-flip TTA on crops\n    SUBMISSION_PATH = (OUT / "submission.csv") if KAGGLE else (SUBMISSIONS_DIR / "submission.csv")\n')

with open("v8_1/geometry.py", "w", encoding="utf-8") as f:
    f.write('"""Shared geometry helpers for Solar Filament instance crop operations.\n\nEnsures strict square bounds without aspect-ratio distortion near image boundaries.\nBoth training (Stage 2) and inference (Stage 3) MUST use this exact contract.\n"""\n\nfrom typing import Sequence, Tuple\n\n\ndef compute_adaptive_crop_side(\n    bbox: Sequence[int],\n    crop_min: int = 256,\n    crop_max: int = 512,\n    crop_padding: float = 1.2,\n) -> int:\n    """Compute adaptive crop side length based on proposal dimensions.\n\n    side = min(crop_max, max(crop_min, int(crop_padding * max(w, h))))\n    """\n    x1, y1, x2, y2 = bbox\n    bw = max(0, int(x2 - x1))\n    bh = max(0, int(y2 - y1))\n    side = int(round(crop_padding * max(bw, bh)))\n    return min(crop_max, max(crop_min, side))\n\n\ndef square_bounds_from_bbox(\n    bbox: Sequence[int],\n    image_height: int,\n    image_width: int,\n    side: int,\n) -> Tuple[int, int, int, int]:\n    """Calculate strictly square bounding coordinates [x1, y1, x2, y2] centered on bbox.\n\n    Invariants guaranteed:\n    1. 0 <= x1 < x2 <= image_width\n    2. 0 <= y1 < y2 <= image_height\n    3. (x2 - x1) == (y2 - y1) == final_side\n    4. Centered on bbox center whenever boundary limits permit.\n    5. When boundary limits clip left/right/top/bottom, shifts the entire window\n       so the crop remains strictly square without aspect-ratio distortion.\n\n    Args:\n        bbox: [x1, y1, x2, y2] bounding box\n        image_height: Source image height H (e.g. 2048)\n        image_width: Source image width W (e.g. 2048)\n        side: Desired crop side length (e.g. 256..512)\n\n    Returns:\n        Tuple of (x1, y1, x2, y2)\n    """\n    final_side = min(int(side), int(image_height), int(image_width))\n    if final_side <= 0:\n        final_side = min(int(image_height), int(image_width))\n\n    bx1, by1, bx2, by2 = bbox\n    cx = (int(bx1) + int(bx2)) // 2\n    cy = (int(by1) + int(by2)) // 2\n\n    # Initial centered bounds\n    x1 = cx - final_side // 2\n    y1 = cy - final_side // 2\n    x2 = x1 + final_side\n    y2 = y1 + final_side\n\n    # Boundary shifts for X\n    if x1 < 0:\n        shift = -x1\n        x1 += shift\n        x2 += shift\n    elif x2 > image_width:\n        shift = x2 - image_width\n        x1 -= shift\n        x2 -= shift\n\n    # Boundary shifts for Y\n    if y1 < 0:\n        shift = -y1\n        y1 += shift\n        y2 += shift\n    elif y2 > image_height:\n        shift = y2 - image_height\n        y1 -= shift\n        y2 -= shift\n\n    # Final sanity clamp\n    x1 = max(0, min(x1, image_width - final_side))\n    x2 = x1 + final_side\n    y1 = max(0, min(y1, image_height - final_side))\n    y2 = y1 + final_side\n\n    return int(x1), int(y1), int(x2), int(y2)\n')

with open("v8_1/1_train_yolo.py", "w", encoding="utf-8") as f:
    f.write('"""\nv8_1 Stage 1 — Train YOLO11s-seg Proposer on MAGFiLO (Frozen Specification).\n\nMaster: ChatGPT | Builder: Antigravity\nTarget: Kaggle Solar Filament Segmentation Challenge 2026\n\nFrozen Rules:\n- Model: yolo11s-seg.pt @ 1024\n- 20 epochs, batch 2, single GPU cuda:0, AMP=True, mosaic=0.5, close_mosaic=2\n- Refuse to start if not torch.cuda.is_available() (unless --dry-run)\n- Multi-annotator validation PQ: evaluate on unique file_names, computing\n  pq_mean and pq_max across annotators. Sweep conf in {0.15, 0.25, 0.35}.\n"""\n\nimport argparse\nimport json\nimport os\nimport sys\nfrom collections import defaultdict\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\nimport torch\n\n# Ensure project root is on sys.path\nPROJECT_ROOT = Path(__file__).resolve().parent.parent\nsys.path.insert(0, str(PROJECT_ROOT))\n\nfrom metrics.pq import pq_score, pq_score_multi\nfrom v8_1.config import YOLOConfig, MAGFILO_DIR, YOLO_DATA_DIR, RUNS_DIR\n\n\nif hasattr(sys.stdout, "reconfigure"):\n    sys.stdout.reconfigure(encoding="utf-8")\n\n\ndef print_frozen_config():\n    """Print the frozen YOLO11s-seg specification for master review."""\n    print("=" * 75)\n    print("V8_1 STAGE 1: YOLO11s-seg PROPOSER (FROZEN CONFIGURATION)")\n    print("=" * 75)\n    print(f"  Model Architecture:    {YOLOConfig.MODEL}")\n    print(f"  Input Resolution:      {YOLOConfig.IMGSZ}x{YOLOConfig.IMGSZ}")\n    print(f"  Epochs:                {YOLOConfig.EPOCHS}")\n    print(f"  Batch Size:            {YOLOConfig.BATCH}")\n    print(f"  Training Device:       cuda:{YOLOConfig.DEVICE} (Single GPU pass 1)")\n    print(f"  Mixed Precision (AMP): {YOLOConfig.AMP}")\n    print(f"  Mosaic Augmentation:   {YOLOConfig.MOSAIC} (close_mosaic: last {YOLOConfig.CLOSE_MOSAIC} epochs)")\n    print(f"  Geometry Augmentation: degrees={YOLOConfig.DEGREES}, fliplr={YOLOConfig.FLIPLR}, flipud={YOLOConfig.FLIPUD}")\n    print(f"  Copy-Paste:            DISABLED (until \'s\' baseline holds)")\n    print(f"  Overlap Mask:          {YOLOConfig.OVERLAP_MASK}")\n    print(f"  Mask Ratio:            {YOLOConfig.MASK_RATIO}")\n    print(f"  Workers:               {YOLOConfig.WORKERS}")\n    print(f"  Dataset Config:        {YOLOConfig.DATA_YAML}")\n    print(f"  Project Run Dir:       {YOLOConfig.PROJECT}/{YOLOConfig.NAME}")\n    print(f"  Target Checkpoint:     {YOLOConfig.WEIGHTS_PATH}")\n    print(f"  CUDA Available:        {torch.cuda.is_available()} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else \'CPU Only\'})")\n    print("=" * 75)\n\n\ndef evaluate_val_pq(model, data_dir: Path, yolo_data_dir: Path, conf_thresholds=(0.15, 0.25, 0.35)):\n    """Evaluate YOLO-seg predictions against unique physical JPEGs.\n\n    For each val file_name:\n    1. Rasterize each annotator observation separately.\n    2. Compute Kirillov PQ vs each annotator.\n    3. Record pq_mean and pq_max.\n    Sweep conf in {0.15, 0.25, 0.35} on pq_mean.\n    """\n    print("\\n" + "=" * 75)\n    print("HOLDOUT MULTI-ANNOTATOR KIRILLOV PQ EVALUATION")\n    print("=" * 75)\n\n    # Load ground truth annotations\n    train_dir = data_dir / "train"\n    json_path = next(train_dir.glob("*.json"))\n    with open(json_path, "r", encoding="utf-8") as f:\n        coco = json.load(f)\n\n    # Map file_name -> list of annotator observation image_ids\n    fn_to_img_ids = defaultdict(list)\n    for img in coco.get("images", []):\n        fn_to_img_ids[img["file_name"]].append(img["id"])\n\n    # Map image_id -> list of polygons (all categories 1, 2, 3, 4 are solar filaments)\n    img_id_to_polys = defaultdict(list)\n    for ann in coco.get("annotations", []):\n        segs = ann.get("segmentation", [])\n        if isinstance(segs, list):\n            for poly in segs:\n                if len(poly) >= 6 and len(poly) % 2 == 0:\n                    img_id_to_polys[ann["image_id"]].append(poly)\n\n    # Identify val files from yolo_data_dir\n    val_label_files = list((yolo_data_dir / "labels" / "val").glob("*.txt"))\n    val_physical_stems = sorted(list({p.stem.split("_ann_")[0] for p in val_label_files}))\n    val_img_dir = data_dir / "train" / "train_images"\n\n    print(f"  Holdout unique physical JPEGs to score: {len(val_physical_stems)}")\n\n    # Record polygon coordinates per val file (streaming, zero RAM overhead)\n    val_gt_by_file = {}\n    for stem in val_physical_stems:\n        fn = f"{stem}.jpeg" if (val_img_dir / f"{stem}.jpeg").exists() else f"{stem}.jpg"\n        img_ids = fn_to_img_ids.get(fn, [])\n        annotator_polys = [img_id_to_polys.get(iid, []) for iid in img_ids]\n        val_gt_by_file[stem] = {"fn": fn, "annotators_polys": annotator_polys}\n\n    results_table = []\n\n    for conf in conf_thresholds:\n        print(f"\\n--- Sweeping Confidence Threshold: {conf:.2f} ---")\n        disk_pq_means, disk_pq_maxs = [], []\n        disk_sq_means, disk_rq_means = [], []\n        tot_tp, tot_fp, tot_fn = 0, 0, 0\n\n        for stem in val_physical_stems:\n            fn = val_gt_by_file[stem]["fn"]\n            img_path = val_img_dir / fn\n            annotators_polys = val_gt_by_file[stem]["annotators_polys"]\n\n            # Stream-rasterize GT masks on-the-fly for this physical image only\n            annotator_instances = []\n            for polys in annotators_polys:\n                masks = []\n                for p in polys:\n                    m = np.zeros((2048, 2048), dtype=np.uint8)\n                    pts = np.asarray(p, dtype=np.int32).reshape(-1, 2)\n                    cv2.fillPoly(m, [pts], 1)\n                    if m.sum() > 0:\n                        masks.append(m)\n                annotator_instances.append(masks)\n\n            # Run YOLO prediction\n            preds = model.predict(\n                source=str(img_path),\n                imgsz=YOLOConfig.IMGSZ,\n                conf=conf,\n                device=0 if torch.cuda.is_available() else "cpu",\n                verbose=False,\n            )\n\n            pred_masks = []\n            if preds and preds[0].masks is not None:\n                # Resize YOLO masks back to 2048x2048\n                raw_masks = preds[0].masks.data.cpu().numpy()\n                for rm in raw_masks:\n                    m_2048 = cv2.resize(rm.astype(np.uint8), (2048, 2048), interpolation=cv2.INTER_NEAREST)\n                    if m_2048.sum() > 0:\n                        pred_masks.append(m_2048)\n\n            # Score against each annotator independently\n            ann_pqs, ann_sqs, ann_rqs = [], [], []\n            for ann_gt in annotator_instances:\n                score_dict = pq_score(pred_masks, ann_gt, iou_threshold=0.5)\n                ann_pqs.append(score_dict["PQ"])\n                ann_sqs.append(score_dict["SQ"])\n                ann_rqs.append(score_dict["RQ"])\n                tot_tp += score_dict["TP"]\n                tot_fp += score_dict["FP"]\n                tot_fn += score_dict["FN"]\n\n            if ann_pqs:\n                disk_pq_means.append(np.mean(ann_pqs))\n                disk_pq_maxs.append(np.max(ann_pqs))\n                disk_sq_means.append(np.mean(ann_sqs))\n                disk_rq_means.append(np.mean(ann_rqs))\n\n            # Release full 2048 masks immediately\n            del pred_masks, annotator_instances\n\n        mean_pq = float(np.mean(disk_pq_means)) if disk_pq_means else 0.0\n        max_pq = float(np.mean(disk_pq_maxs)) if disk_pq_maxs else 0.0\n        mean_sq = float(np.mean(disk_sq_means)) if disk_sq_means else 0.0\n        mean_rq = float(np.mean(disk_rq_means)) if disk_rq_means else 0.0\n        results_table.append({\n            "conf": conf,\n            "pq_mean": mean_pq,\n            "pq_max": max_pq,\n            "sq_mean": mean_sq,\n            "rq_mean": mean_rq,\n            "tp": tot_tp,\n            "fp": tot_fp,\n            "fn": tot_fn,\n        })\n        print(f"  Confidence {conf:.2f} -> pq_mean: {mean_pq:.4f} | pq_max: {max_pq:.4f} | SQ: {mean_sq:.4f} | RQ: {mean_rq:.4f} (TP={tot_tp}, FP={tot_fp}, FN={tot_fn})")\n\n    print("\\n" + "=" * 85)\n    print("SWEEP SUMMARY TABLE (Unique Physical Disks)")\n    print("=" * 85)\n    print(f"{\'CONFIDENCE\':<12} | {\'pq_mean (Selector)\':<20} | {\'pq_max\':<12} | {\'SQ\':<10} | {\'RQ\':<10} | {\'TP/FP/FN\'}")\n    print("-" * 85)\n    for r in results_table:\n        print(f"{r[\'conf\']:<12.2f} | {r[\'pq_mean\']:<20.4f} | {r[\'pq_max\']:<12.4f} | {r[\'sq_mean\']:<10.4f} | {r[\'rq_mean\']:<10.4f} | {r[\'tp\']}/{r[\'fp\']}/{r[\'fn\']}")\n    print("=" * 85)\n\n    best_conf = max(results_table, key=lambda x: x["pq_mean"])\n    print(f"[OPTIMAL] Optimal Threshold by pq_mean: conf={best_conf[\'conf\']} (pq_mean={best_conf[\'pq_mean\']:.4f})")\n    return results_table\n    return results_table\n\n\ndef main():\n    parser = argparse.ArgumentParser(description="Train YOLO11s-seg Stage 1 Proposer")\n    parser.add_argument("--dry-run", action="store_true", help="Print frozen config and exit 0 without training")\n    parser.add_argument("--eval-only", action="store_true", help="Run holdout PQ evaluation only using best.pt")\n    parser.add_argument("--max-train", type=int, default=None, help="Limit training samples for 1-epoch GPU smoke test")\n    parser.add_argument("--batch", type=int, default=YOLOConfig.BATCH, help="Batch size (drop to 2 on T4 if OOM)")\n    parser.add_argument("--epochs", type=int, default=YOLOConfig.EPOCHS, help="Number of training epochs")\n    parser.add_argument("--weights", type=str, default=None, help="Path to existing weights for evaluation")\n    args = parser.parse_args()\n\n    # Step 1: Handle --dry-run\n    if args.dry_run:\n        print_frozen_config()\n        print("Dry run completed successfully. Exiting 0 on CPU.\\n")\n        sys.exit(0)\n\n    # Step 2: Enforce CUDA requirement\n    if not torch.cuda.is_available():\n        print("[ERROR] CUDA is not available! YOLO11s-seg training refuses to run on CPU.")\n        print("Run with --dry-run to inspect configuration on CPU.")\n        sys.exit(1)\n\n    try:\n        from ultralytics import YOLO\n    except ImportError:\n        print("[ERROR] \'ultralytics\' package is not installed.")\n        print("Install via: pip install ultralytics")\n        sys.exit(1)\n\n    print_frozen_config()\n\n    # Step 3: Handle --eval-only\n    if args.eval_only:\n        ckpt = Path(args.weights) if args.weights else YOLOConfig.WEIGHTS_PATH\n        if not ckpt.exists():\n            print(f"[ERROR] Checkpoint not found at {ckpt}")\n            sys.exit(1)\n        model = YOLO(str(ckpt))\n        evaluate_val_pq(model, data_dir=MAGFILO_DIR, yolo_data_dir=YOLO_DATA_DIR)\n        return\n\n    # Step 4: Launch Training\n    print("[INFO] Launching YOLO11s-seg Training on GPU...")\n    model = YOLO(YOLOConfig.MODEL)\n\n    data_yaml = str(YOLOConfig.DATA_YAML)\n    if not Path(data_yaml).exists():\n        print(f"[ERROR] Dataset yaml not found at {data_yaml}. Run convert_coco_to_yolo.py first.")\n        sys.exit(1)\n\n    train_kwargs = {\n        "data": data_yaml,\n        "epochs": args.epochs,\n        "imgsz": YOLOConfig.IMGSZ,\n        "batch": args.batch,\n        "amp": YOLOConfig.AMP,\n        "mosaic": YOLOConfig.MOSAIC,\n        "close_mosaic": YOLOConfig.CLOSE_MOSAIC,\n        "degrees": YOLOConfig.DEGREES,\n        "flipud": YOLOConfig.FLIPUD,\n        "fliplr": YOLOConfig.FLIPLR,\n        "workers": YOLOConfig.WORKERS,\n        "project": YOLOConfig.PROJECT,\n        "name": YOLOConfig.NAME,\n        "device": YOLOConfig.DEVICE,\n        "overlap_mask": YOLOConfig.OVERLAP_MASK,\n        "mask_ratio": YOLOConfig.MASK_RATIO,\n        "exist_ok": True,\n        "save": True,\n        "plots": False,\n    }\n\n    if args.max_train:\n        print(f"  [SMOKE] Limiting to {args.max_train} training samples for quick validation.")\n        train_kwargs["fraction"] = min(1.0, args.max_train / 951.0)\n\n    results = model.train(**train_kwargs)\n    print("[OK] YOLO11s-seg training run completed!")\n\n    # Memory cleanup & VRAM logging\n    del model, results\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n        for dev_idx in range(torch.cuda.device_count()):\n            res_mb = torch.cuda.memory_reserved(dev_idx) / (1024 ** 2)\n            alloc_mb = torch.cuda.memory_allocated(dev_idx) / (1024 ** 2)\n            print(f"  [GPU {dev_idx}] Memory Reserved: {res_mb:.1f} MB | Allocated: {alloc_mb:.1f} MB")\n\n    # Step 5: Post-train holdout PQ evaluation\n    best_weights = YOLOConfig.resolve_weights(args.weights)\n    if best_weights.exists():\n        print(f"  [BEST WEIGHTS LOCATED]: {best_weights}")\n        trained_model = YOLO(str(best_weights))\n        evaluate_val_pq(trained_model, data_dir=MAGFILO_DIR, yolo_data_dir=YOLO_DATA_DIR)\n        del trained_model\n        torch.cuda.empty_cache()\n    else:\n        print(f"[WARN] Best weights not found at {best_weights}, skipping PQ evaluation.")\n\n\nif __name__ == "__main__":\n    main()\n')

with open("v8_1/2_train_crop_refiner.py", "w", encoding="utf-8") as f:
    f.write('"""\nv8_1 Stage 2 — Train Crop U-Net Refiner (ResNet-34) on MAGFiLO Crops.\n\nMaster: ChatGPT | Builder: Antigravity\nTarget: Kaggle Solar Filament Segmentation Challenge 2026\n\nFrozen Rules:\n- Train only on train-fold instances (579 files / 951 observations).\n- Crop side: min(512, max(256, int(1.2 * max(w, h)))), square resize to 384 for batch.\n- 3-channel input: [Raw Grayscale, CLAHE (3.0, 8x8), High-Pass Unsharp Filter].\n- Model: U-Net with ResNet-34 backbone, 1 class, AMP.\n- Refuse to train if not torch.cuda.is_available() (unless --dry-run).\n- Require YOLO best.pt unless --boxes-from-gt (allowed for parallel GT-crop pretrain).\n"""\n\nimport argparse\nimport json\nimport os\nimport sys\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\nimport torch\nimport torch.nn as nn\nfrom torch.utils.data import Dataset, DataLoader\n\n# Ensure project root is on sys.path\nPROJECT_ROOT = Path(__file__).resolve().parent.parent\nsys.path.insert(0, str(PROJECT_ROOT))\n\nfrom v8_1.config import CropRefinerConfig, YOLOConfig, MAGFILO_DIR, YOLO_DATA_DIR, MODELS_DIR\nfrom v8_1.geometry import compute_adaptive_crop_side, square_bounds_from_bbox\n\nif hasattr(sys.stdout, "reconfigure"):\n    sys.stdout.reconfigure(encoding="utf-8")\n\n\ndef print_crop_refiner_config(boxes_from_gt: bool):\n    """Print the frozen Crop Refiner specification for review."""\n    print("=" * 75)\n    print("V8_1 STAGE 2: CROP U-NET REFINER (FROZEN CONFIGURATION)")\n    print("=" * 75)\n    print(f"  Backbone Architecture: {CropRefinerConfig.ENCODER} (U-Net, 1 class)")\n    print(f"  Input Channels:        {CropRefinerConfig.IN_CHANNELS} ([Raw, CLAHE, Unsharp])")\n    print(f"  Square Crop Target:    {CropRefinerConfig.TARGET_SIZE}x{CropRefinerConfig.TARGET_SIZE} (Square Resize)")\n    print(f"  Adaptive Crop Formula: min({CropRefinerConfig.CROP_MAX}, max({CropRefinerConfig.CROP_MIN}, int({CropRefinerConfig.CROP_PADDING} * max(w, h))))")\n    print(f"  Epochs:                {CropRefinerConfig.EPOCHS}")\n    print(f"  Batch Size:            {CropRefinerConfig.BATCH_SIZE}")\n    print(f"  Learning Rate:         {CropRefinerConfig.LR} (AdamW, Cosine Annealing)")\n    print(f"  Mixed Precision (AMP): {CropRefinerConfig.AMP}")\n    print(f"  Box Proposals Source:  {\'Ground Truth Polygons (--boxes-from-gt)\' if boxes_from_gt else \'YOLO11s-seg best.pt\'}")\n    print(f"  Output Checkpoint:     {CropRefinerConfig.CKPT_PATH}")\n    print(f"  CUDA Available:        {torch.cuda.is_available()} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else \'CPU Only\'})")\n    print("=" * 75)\n\n\ndef preprocess_3ch(gray: np.ndarray) -> np.ndarray:\n    """Build [Raw, CLAHE, Unsharp] 3-channel input."""\n    ch_raw = gray.copy()\n    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))\n    ch_clahe = clahe.apply(gray)\n    blur = cv2.GaussianBlur(gray, (0, 0), sigmaX=5)\n    ch_unsharp = cv2.addWeighted(gray, 1.5, blur, -0.5, 0)\n    return np.stack([ch_raw, ch_clahe, ch_unsharp], axis=-1)\n\n\nclass FilamentCropDataset(Dataset):\n    """Lazy-loaded object-centered crop dataset for Stage 2 Refiner."""\n\n    def __init__(self, crop_records: list[dict], target_size: int = 384, augment: bool = True):\n        self.records = crop_records\n        self.target_size = target_size\n        self.augment = augment\n\n    def __len__(self):\n        return len(self.records)\n\n    def __getitem__(self, idx):\n        rec = self.records[idx]\n        img_path = rec["img_path"]\n        bbox = rec["bbox"]  # [x1, y1, x2, y2]\n        pts = rec["pts"]    # polygon points (N, 2)\n\n        # 1. Read grayscale image and preprocess\n        gray = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)\n        if gray is None:\n            raise RuntimeError(f"FATAL: Unable to decode training image: {img_path}")\n        img_3ch = preprocess_3ch(gray)\n        h, w = gray.shape[:2]\n\n        # 2. Rasterize polygon instance mask\n        mask = np.zeros((h, w), dtype=np.uint8)\n        cv2.fillPoly(mask, [pts.astype(np.int32)], 1)\n\n        # 3. Strictly square adaptive crop window (no aspect-ratio distortion near boundaries)\n        side = compute_adaptive_crop_side(\n            bbox,\n            crop_min=CropRefinerConfig.CROP_MIN,\n            crop_max=CropRefinerConfig.CROP_MAX,\n            crop_padding=CropRefinerConfig.CROP_PADDING,\n        )\n        crop_x1, crop_y1, crop_x2, crop_y2 = square_bounds_from_bbox(bbox, h, w, side)\n\n        crop_img = img_3ch[crop_y1:crop_y2, crop_x1:crop_x2]\n        crop_mask = mask[crop_y1:crop_y2, crop_x1:crop_x2]\n\n        # 4. Square resize to target_size (384x384)\n        resized_img = cv2.resize(crop_img, (self.target_size, self.target_size), interpolation=cv2.INTER_LINEAR)\n        resized_mask = cv2.resize(crop_mask, (self.target_size, self.target_size), interpolation=cv2.INTER_NEAREST)\n\n        # 5. Optional online flip augmentation\n        if self.augment:\n            if np.random.rand() > 0.5:\n                resized_img = np.fliplr(resized_img).copy()\n                resized_mask = np.fliplr(resized_mask).copy()\n            if np.random.rand() > 0.5:\n                resized_img = np.flipud(resized_img).copy()\n                resized_mask = np.flipud(resized_mask).copy()\n\n        # Convert to tensor: (C, H, W) normalized to [0, 1]\n        img_tensor = torch.from_numpy(resized_img).permute(2, 0, 1).float() / 255.0\n        mask_tensor = torch.from_numpy(resized_mask).unsqueeze(0).float()\n\n        return img_tensor, mask_tensor\n\n\ndef build_crop_records(data_dir: Path, yolo_data_dir: Path) -> list[dict]:\n    """Build crop metadata records strictly from train-fold instances."""\n    train_label_files = list((yolo_data_dir / "labels" / "train").glob("*.txt"))\n    train_physical_stems = {p.stem.split("_ann_")[0] for p in train_label_files}\n\n    train_dir = data_dir / "train"\n    json_path = next(train_dir.glob("*.json"))\n    with open(json_path, "r", encoding="utf-8") as f:\n        coco = json.load(f)\n\n    id_to_file = {img["id"]: img["file_name"] for img in coco.get("images", [])}\n    records = []\n\n    for ann in coco.get("annotations", []):\n        # All categories 1, 2, 3, 4 are solar filaments for segmentation objective\n        img_id = ann["image_id"]\n        fn = id_to_file.get(img_id)\n        if not fn:\n            continue\n        stem = Path(fn).stem\n        if stem not in train_physical_stems:\n            continue  # Must stay strictly inside train fold\n\n        segs = ann.get("segmentation", [])\n        if not isinstance(segs, list):\n            continue\n\n        img_path = data_dir / "train" / "train_images" / fn\n        for poly in segs:\n            if isinstance(poly, list) and len(poly) >= 6 and len(poly) % 2 == 0:\n                pts = np.asarray(poly, dtype=np.float32).reshape(-1, 2)\n                x1, y1 = np.min(pts, axis=0)\n                x2, y2 = np.max(pts, axis=0)\n                if (x2 - x1) > 2 and (y2 - y1) > 2:\n                    records.append({\n                        "img_path": img_path,\n                        "bbox": [int(x1), int(y1), int(x2), int(y2)],\n                        "pts": pts,\n                    })\n\n    return records\n\n\ndef train_crop_refiner(records: list[dict]):\n    """Execute training loop for Crop U-Net refiner on GPU."""\n    import segmentation_models_pytorch as smp\n\n    print(f"📦 Assembling Crop Dataset ({len(records)} instance crops)...")\n    dataset = FilamentCropDataset(records, target_size=CropRefinerConfig.TARGET_SIZE, augment=True)\n    loader = DataLoader(\n        dataset,\n        batch_size=CropRefinerConfig.BATCH_SIZE,\n        shuffle=True,\n        num_workers=2 if os.name != "nt" else 0,\n        pin_memory=True,\n    )\n\n    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")\n    model = smp.Unet(\n        encoder_name=CropRefinerConfig.ENCODER,\n        encoder_weights="imagenet",\n        in_channels=CropRefinerConfig.IN_CHANNELS,\n        classes=CropRefinerConfig.CLASSES,\n    ).to(device)\n\n    optimizer = torch.optim.AdamW(model.parameters(), lr=CropRefinerConfig.LR, weight_decay=CropRefinerConfig.WEIGHT_DECAY)\n    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=CropRefinerConfig.EPOCHS, eta_min=1e-6)\n    bce_loss_fn = nn.BCEWithLogitsLoss()\n    scaler = torch.amp.GradScaler("cuda", enabled=CropRefinerConfig.AMP)\n\n    print("🚀 Training Crop U-Net Refiner...")\n    model.train()\n\n    import time\n    epoch_durations = []\n    t_start = time.perf_counter()\n\n    for epoch in range(1, CropRefinerConfig.EPOCHS + 1):\n        t_ep_start = time.perf_counter()\n        total_loss = 0.0\n        for imgs, masks in loader:\n            imgs, masks = imgs.to(device), masks.to(device)\n            optimizer.zero_grad()\n\n            with torch.amp.autocast("cuda", enabled=CropRefinerConfig.AMP):\n                logits = model(imgs)\n                # Combined BCE + Dice loss\n                probs = torch.sigmoid(logits)\n                intersection = (probs * masks).sum(dim=(1, 2, 3))\n                dice = (2.0 * intersection + 1.0) / (probs.sum(dim=(1, 2, 3)) + masks.sum(dim=(1, 2, 3)) + 1.0)\n                loss = 0.5 * bce_loss_fn(logits, masks) + 0.5 * (1.0 - dice.mean())\n\n            scaler.scale(loss).backward()\n            scaler.step(optimizer)\n            scaler.update()\n            total_loss += loss.item()\n\n        scheduler.step()\n        ep_duration = time.perf_counter() - t_ep_start\n        epoch_durations.append(ep_duration)\n        avg_loss = total_loss / max(1, len(loader))\n        print(f"  Epoch [{epoch:02d}/{CropRefinerConfig.EPOCHS:02d}] - Loss: {avg_loss:.4f} - LR: {scheduler.get_last_lr()[0]:.6f} - Duration: {ep_duration:.2f}s")\n\n    total_training_time = time.perf_counter() - t_start\n    avg_epoch_duration = sum(epoch_durations) / len(epoch_durations) if epoch_durations else 0.0\n    print(f"\\n[TIMING] Crop Refiner Average Epoch Duration: {avg_epoch_duration:.2f}s | Total: {total_training_time:.2f}s ({total_training_time / 60:.2f} min)")\n\n    # Save checkpoint\n    CropRefinerConfig.CKPT_PATH.parent.mkdir(parents=True, exist_ok=True)\n    torch.save(model.state_dict(), CropRefinerConfig.CKPT_PATH)\n    print(f"[OK] Saved Crop U-Net refiner checkpoint to: {CropRefinerConfig.CKPT_PATH}")\n\n    # Memory cleanup & VRAM logging\n    del model, optimizer, scheduler, loader, dataset\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n        for dev_idx in range(torch.cuda.device_count()):\n            res_mb = torch.cuda.memory_reserved(dev_idx) / (1024 ** 2)\n            alloc_mb = torch.cuda.memory_allocated(dev_idx) / (1024 ** 2)\n            print(f"  [GPU {dev_idx}] Memory Reserved: {res_mb:.1f} MB | Allocated: {alloc_mb:.1f} MB")\n\n\ndef main():\n    parser = argparse.ArgumentParser(description="Train Crop U-Net Refiner")\n    parser.add_argument("--dry-run", action="store_true", help="Print frozen config and exit 0 without training")\n    parser.add_argument("--boxes-from-gt", action="store_true", help="Extract crops directly from GT polygons for parallel pretrain")\n    args = parser.parse_args()\n\n    if args.dry_run:\n        print_crop_refiner_config(boxes_from_gt=args.boxes_from_gt)\n        print("Dry run completed successfully. Exiting 0 on CPU.\\n")\n        sys.exit(0)\n\n    if not torch.cuda.is_available():\n        print("[ERROR] CUDA is not available! Crop Refiner training refuses to run on CPU.")\n        print("Run with --dry-run to inspect configuration on CPU.")\n        sys.exit(1)\n\n    # Dependency check: require YOLO weights unless --boxes-from-gt is set\n    if not args.boxes_from_gt and not YOLOConfig.WEIGHTS_PATH.exists():\n        print(f"[ERROR] Stage 1 YOLO weights not found at {YOLOConfig.WEIGHTS_PATH}!")\n        print("Train Stage 1 YOLO first, or specify --boxes-from-gt to run a parallel GT-crop pretrain.")\n        sys.exit(1)\n\n    print_crop_refiner_config(boxes_from_gt=args.boxes_from_gt)\n\n    records = build_crop_records(data_dir=MAGFILO_DIR, yolo_data_dir=YOLO_DATA_DIR)\n    print(f"  Loaded {len(records)} instance crops strictly from train fold (579 files).")\n    train_crop_refiner(records)\n\n\nif __name__ == "__main__":\n    main()\n')

with open("v8_1/3_infer_cascade.py", "w", encoding="utf-8") as f:
    f.write('"""\nv8_1 Stage 3 — Instance-Segmentation Cascade Inference Engine.\n\nMaster: ChatGPT | Builder: Antigravity\nTarget: Kaggle Solar Filament Segmentation Challenge 2026\n\nFrozen Rules:\n- Pipeline: YOLO proposals -> adaptive pad -> crop refiner (4-flip TTA) -> residual (IoU<0.30) -> greedy score non-overlap\n- Residual OFF by default (--residual 0).\n- Adaptive crop formula: min(512, max(256, int(1.2 * max(w, h)))).\n- 4-flip TTA on crops only.\n- Empty disk: 0 rows (strictly adheres to host contract; every row represents a real filament).\n- Output: submissions/v8_1_cascade.csv.\n- Statistics audit: n_images, n_rows, rows/image (mean, p50, p90), n_empty, elapsed seconds.\n"""\n\nimport argparse\nimport csv\nimport os\nimport sys\nimport time\nfrom collections import defaultdict\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\nimport torch\n\n# Ensure project root is on sys.path\nPROJECT_ROOT = Path(__file__).resolve().parent.parent\nsys.path.insert(0, str(PROJECT_ROOT))\n\nfrom metrics.pq import encode_mask, decode_rle\nfrom v8_1.config import CascadeConfig, CropRefinerConfig, YOLOConfig, MAGFILO_DIR, SUBMISSIONS_DIR\nfrom v8_1.geometry import compute_adaptive_crop_side, square_bounds_from_bbox\n\nif hasattr(sys.stdout, "reconfigure"):\n    sys.stdout.reconfigure(encoding="utf-8")\n\n\ndef print_infer_config(args):\n    """Print the frozen Inference Cascade specification for review."""\n    print("=" * 75)\n    print("V8_1 STAGE 3: CASCADE INFERENCE (CONFIGURATION)")\n    print("=" * 75)\n    print(f"  Stage 1 YOLO Proposer: {YOLOConfig.WEIGHTS_PATH} (conf={args.conf})")\n    print(f"  Stage 2 Crop Refiner:  {CropRefinerConfig.CKPT_PATH} (4-flip TTA={args.tta})")\n    print(f"  Residual Discovery:    DISABLED (Pure YOLO -> Crop Refiner cascade)")\n    print(f"  Solar Disk Mask:       Radius = {CascadeConfig.SOLAR_DISK_R_FRAC} * 1024 (~952 px)")\n    print(f"  Min Area Threshold:    {getattr(args, \'min_area\', 100)} px")\n    print(f"  YOLO Mask Fallback:    {getattr(args, \'yolo_fallback\', 0)}")\n    print(f"  Overlap Strategy:      {getattr(args, \'overlap_mode\', \'trim\')}")\n    print(f"  Empty Disk Policy:     0 rows emitted (per official host PQ metric contract)")\n    print(f"  Output Submission:     {args.out}")\n    print(f"  CUDA Available:        {torch.cuda.is_available()} ({torch.cuda.get_device_name(0) if torch.cuda.is_available() else \'CPU Only\'})")\n    print("=" * 75)\n\n\ndef get_solar_disk_mask(h: int = 2048, w: int = 2048, r_frac: float = 0.93) -> np.ndarray:\n    """Generate circular solar disk mask centered at (w/2, h/2)."""\n    mask = np.zeros((h, w), dtype=np.uint8)\n    cx, cy = w // 2, h // 2\n    r = int(r_frac * min(h, w) / 2.0)\n    cv2.circle(mask, (cx, cy), r, 1, -1)\n    return mask\n\n\ndef preprocess_3ch(gray: np.ndarray) -> np.ndarray:\n    """Build [Raw, CLAHE, Unsharp] 3-channel input."""\n    ch_raw = gray.copy()\n    clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))\n    ch_clahe = clahe.apply(gray)\n    blur = cv2.GaussianBlur(gray, (0, 0), sigmaX=5)\n    ch_unsharp = cv2.addWeighted(gray, 1.5, blur, -0.5, 0)\n    return np.stack([ch_raw, ch_clahe, ch_unsharp], axis=-1)\n\n\ndef refine_crop_tta(\n    refiner_model,\n    img_3ch: np.ndarray,\n    bbox: list[int],\n    device: torch.device,\n    target_size: int = 384,\n    use_tta: bool = True,\n    threshold: float = 0.50,\n) -> np.ndarray:\n    """Refine a single candidate using Crop U-Net with 4-flip TTA."""\n    h, w = img_3ch.shape[:2]\n\n    # Strictly square adaptive crop window (identical contract to training)\n    side = compute_adaptive_crop_side(\n        bbox,\n        crop_min=CropRefinerConfig.CROP_MIN,\n        crop_max=CropRefinerConfig.CROP_MAX,\n        crop_padding=CropRefinerConfig.CROP_PADDING,\n    )\n    c_x1, c_y1, c_x2, c_y2 = square_bounds_from_bbox(bbox, h, w, side)\n\n    crop_img = img_3ch[c_y1:c_y2, c_x1:c_x2]\n    orig_ch, orig_cw = crop_img.shape[:2]\n\n    # Square resize to target_size\n    resized = cv2.resize(crop_img, (target_size, target_size), interpolation=cv2.INTER_LINEAR)\n    inp = torch.from_numpy(resized).permute(2, 0, 1).float().unsqueeze(0) / 255.0\n    inp = inp.to(device)\n\n    refiner_model.eval()\n    with torch.no_grad():\n        if use_tta:\n            # 4-flip TTA: [original, fliplr, flipud, fliplr+flipud]\n            p0 = torch.sigmoid(refiner_model(inp))\n            p1 = torch.sigmoid(refiner_model(torch.flip(inp, dims=[3])))\n            p1 = torch.flip(p1, dims=[3])\n            p2 = torch.sigmoid(refiner_model(torch.flip(inp, dims=[2])))\n            p2 = torch.flip(p2, dims=[2])\n            p3 = torch.sigmoid(refiner_model(torch.flip(inp, dims=[2, 3])))\n            p3 = torch.flip(p3, dims=[2, 3])\n            prob_map = (p0 + p1 + p2 + p3) / 4.0\n        else:\n            prob_map = torch.sigmoid(refiner_model(inp))\n\n    prob_np = prob_map.squeeze().cpu().numpy()\n    crop_prob = cv2.resize(prob_np, (orig_cw, orig_ch), interpolation=cv2.INTER_LINEAR)\n    bin_crop = (crop_prob > threshold).astype(np.uint8)\n\n    # Paste back into full 2048x2048 canvas\n    full_mask = np.zeros((h, w), dtype=np.uint8)\n    full_mask[c_y1:c_y2, c_x1:c_x2] = bin_crop\n    return full_mask\n\n\ndef sanitize_instances_zero_overlap(\n    instances: list[dict],\n    min_area: int = 400,\n) -> list[dict]:\n    """Strict host-contract sanitizer: guarantees ZERO shared pixels per disk.\n    \n    1. Sort accepted instances by confidence descending, then area descending.\n    2. Greedy pixel carve: mask[occupied > 0] = 0.\n    3. Drop if area < min_area after carve.\n    4. Update occupied buffer.\n    5. Per disk ASSERT: sum(mask_i & mask_j) == 0 for all i != j.\n       Fail the notebook immediately if any overlap remains.\n    """\n    if not instances:\n        return []\n\n    # 1. Sort by confidence desc, then area desc\n    sorted_inst = sorted(\n        instances,\n        key=lambda x: (float(x["confidence"]), int(x["mask"].sum())),\n        reverse=True,\n    )\n\n    occupied = None\n    sanitized = []\n\n    for inst in sorted_inst:\n        m = inst["mask"].copy()\n        if occupied is not None:\n            m[occupied > 0] = 0\n\n        area = int(m.sum())\n        if area < min_area:\n            continue\n\n        if occupied is None:\n            occupied = (m > 0).astype(np.uint8)\n        else:\n            occupied = np.maximum(occupied, (m > 0).astype(np.uint8))\n\n        sanitized.append({\n            "confidence": float(inst["confidence"]),\n            "mask": m,\n        })\n\n    # 5. Strict Pairwise & Union Disjoint Assertions\n    n = len(sanitized)\n    for i in range(n):\n        for j in range(i + 1, n):\n            overlap = int(np.logical_and(sanitized[i]["mask"], sanitized[j]["mask"]).sum())\n            assert overlap == 0, f"FATAL: Overlap detected between instance {i} and {j}: {overlap} pixels!"\n\n    if n > 0 and occupied is not None:\n        disjoint_sum = sum(int(inst["mask"].sum()) for inst in sanitized)\n        union_sum = int(occupied.sum())\n        assert disjoint_sum == union_sum, f"FATAL: Disjoint sum mismatch: disjoint={disjoint_sum} vs union={union_sum}!"\n\n    return sanitized\n\n\ndef arbitrate_instances(\n    candidates: list[dict],\n    disk_mask: np.ndarray = None,\n    min_area: int = 100,\n    overlap_mode: str = "trim",\n) -> list[dict]:\n    """Arbitrate candidate instances sorted by confidence descending.\n    \n    Args:\n        candidates: list of dicts with \'confidence\' and \'mask\'.\n        disk_mask: circular mask (h, w) for solar disk clipping.\n        min_area: minimum pixel area threshold.\n        overlap_mode: \'trim\' (greedy pixel carve) or \'allow\' (instance NMS at IoU>0.5).\n    """\n    sorted_cands = sorted(candidates, key=lambda x: x["confidence"], reverse=True)\n    occupied = None\n    accepted_instances = []\n\n    for c in sorted_cands:\n        mask = c["mask"].copy()\n        if disk_mask is not None:\n            mask[disk_mask == 0] = 0\n\n        if mask.sum() < min_area:\n            continue\n\n        if overlap_mode == "trim":\n            if occupied is not None:\n                mask[occupied > 0] = 0\n            if mask.sum() < min_area:\n                continue\n            if occupied is None:\n                occupied = mask.copy()\n            else:\n                occupied = np.maximum(occupied, mask)\n            accepted_instances.append({\n                "confidence": c["confidence"],\n                "mask": mask,\n            })\n        elif overlap_mode == "allow":\n            # Instance-level NMS at IoU > 0.5 without carving valid pixels\n            suppress = False\n            for accepted in accepted_instances:\n                acc_mask = accepted["mask"]\n                intersection = np.logical_and(mask, acc_mask).sum()\n                if intersection > 0:\n                    union = np.logical_or(mask, acc_mask).sum()\n                    iou = float(intersection) / float(union) if union > 0 else 0.0\n                    if iou > 0.5:\n                        suppress = True\n                        break\n            if not suppress:\n                accepted_instances.append({\n                    "confidence": c["confidence"],\n                    "mask": mask,\n                })\n        else:\n            raise ValueError(f"Unknown overlap_mode: {overlap_mode}")\n\n    return accepted_instances\n\n\n# Legacy alias for backward compatibility\ngreedy_non_overlap_arbitration = arbitrate_instances\n\n\ndef run_cascade_inference(args):\n    """Execute end-to-end inference cascade across test images."""\n    test_img_dir = MAGFILO_DIR / "test" / "test_images"\n    test_jpegs = sorted(list(test_img_dir.glob("*.jpeg")) + list(test_img_dir.glob("*.jpg")))\n    if not test_jpegs:\n        raise FileNotFoundError(f"No test images found in {test_img_dir}")\n\n    # Dual-T4 Model Parallelism: YOLO on cuda:0, Crop U-Net on cuda:1 (if available)\n    n_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0\n    device_yolo = 0 if n_gpus > 0 else "cpu"\n    refiner_gpu_idx = 1 if n_gpus >= 2 else (0 if n_gpus == 1 else "cpu")\n    device_refiner = torch.device(f"cuda:{refiner_gpu_idx}" if n_gpus > 0 else "cpu")\n    disk_mask = get_solar_disk_mask(2048, 2048, r_frac=CascadeConfig.SOLAR_DISK_R_FRAC)\n\n    print(f"  Target test disks to process: {len(test_jpegs)}")\n    print(f"  Model-Parallel Placement: YOLO -> cuda:{device_yolo} | Refiner -> {device_refiner}")\n\n    # Load Stage 1 YOLO model\n    from ultralytics import YOLO\n    yolo_ckpt = YOLOConfig.resolve_weights(args.weights if hasattr(args, "weights") else None)\n    if not yolo_ckpt.exists():\n        raise FileNotFoundError(f"Stage 1 YOLO weights not found at {yolo_ckpt}")\n    print(f"  Stage 1 YOLO Proposer loaded from: {yolo_ckpt}")\n    yolo_model = YOLO(str(yolo_ckpt))\n\n    # Load Stage 2 Crop U-Net refiner\n    import segmentation_models_pytorch as smp\n    refiner_ckpt = CropRefinerConfig.resolve_ckpt(args.refiner_weights if hasattr(args, "refiner_weights") else None)\n    if not refiner_ckpt.exists():\n        raise FileNotFoundError(f"Stage 2 Crop Refiner checkpoint not found at {refiner_ckpt}")\n    print(f"  Stage 2 Crop Refiner loaded from: {refiner_ckpt}")\n\n    refiner_model = smp.Unet(\n        encoder_name=CropRefinerConfig.ENCODER,\n        in_channels=CropRefinerConfig.IN_CHANNELS,\n        classes=CropRefinerConfig.CLASSES,\n    ).to(device_refiner)\n    refiner_model.load_state_dict(torch.load(refiner_ckpt, map_location=device_refiner))\n    refiner_model.eval()\n\n    all_rows = []\n    rows_per_disk = []\n    empty_disk_count = 0\n    t0 = time.perf_counter()\n\n    min_area = getattr(args, "min_area", 400)\n    yolo_fallback = getattr(args, "yolo_fallback", 0)\n    overlap_mode = getattr(args, "overlap_mode", "trim")\n\n    if overlap_mode != "trim":\n        raise ValueError(\n            f"FATAL: overlap_mode=\'{overlap_mode}\' is FORBIDDEN for submission generation! "\n            f"Host strictly requires ZERO shared pixels between predicted masks. Must use overlap_mode=\'trim\'."\n        )\n\n    for idx, p in enumerate(test_jpegs):\n        stem = p.stem\n        raw = cv2.imread(str(p), cv2.IMREAD_GRAYSCALE)\n        if raw is None:\n            raise RuntimeError(f"FATAL: Unable to decode test image at {p}! Official test image is corrupt or missing.")\n\n        h, w = raw.shape[:2]\n        img_3ch = preprocess_3ch(raw)\n\n        # 1. YOLO Proposals (Input is 2048 JPEG; Ultralytics maps boxes back to 2048 source frame)\n        yolo_preds = yolo_model.predict(\n            source=str(p),\n            imgsz=YOLOConfig.IMGSZ,\n            conf=args.conf,\n            device=device_yolo,\n            verbose=False,\n        )\n\n        candidates = []\n        if yolo_preds and yolo_preds[0].boxes is not None and len(yolo_preds[0].boxes) > 0:\n            boxes = yolo_preds[0].boxes.xyxy.cpu().numpy()\n            confs = yolo_preds[0].boxes.conf.cpu().numpy()\n            yolo_masks_raw = yolo_preds[0].masks.data.cpu().numpy() if (yolo_preds and yolo_preds[0].masks is not None) else None\n\n            # One-line debug check on first image to guarantee boxes are in 2048 space\n            if idx == 0 and len(boxes) > 0:\n                min_c = float(boxes.min())\n                max_c = float(boxes.max())\n                print(f"  [DEBUG Image 0] YOLO raw xyxy range: min={min_c:.1f}, max={max_c:.1f} (assert <= {max(h, w)})")\n                assert max_c <= max(h, w) + 50, f"FATAL: YOLO box coordinates exceed image bounds! Got max={max_c}"\n\n            for b_idx, (box, conf) in enumerate(zip(boxes, confs)):\n                # Clip directly to native 2048 coordinate frame (DO NOT multiply by scale!)\n                x1 = max(0, int(box[0]))\n                y1 = max(0, int(box[1]))\n                x2 = min(w, int(box[2]))\n                y2 = min(h, int(box[3]))\n                bbox_2048 = [x1, y1, x2, y2]\n\n                if (x2 - x1) < 2 or (y2 - y1) < 2:\n                    continue\n\n                # Refine with Crop U-Net on device_refiner\n                refined_mask = refine_crop_tta(\n                    refiner_model=refiner_model,\n                    img_3ch=img_3ch,\n                    bbox=bbox_2048,\n                    device=device_refiner,\n                    target_size=CropRefinerConfig.TARGET_SIZE,\n                    use_tta=args.tta,\n                )\n\n                # YOLO mask fallback: if refiner mask is below min_area, use YOLO-seg mask if valid\n                if yolo_fallback == 1 and refined_mask.sum() < min_area:\n                    if yolo_masks_raw is not None and b_idx < len(yolo_masks_raw):\n                        yolo_m = cv2.resize(yolo_masks_raw[b_idx].astype(np.uint8), (w, h), interpolation=cv2.INTER_NEAREST)\n                        if yolo_m.sum() >= min_area:\n                            refined_mask = yolo_m\n\n                if refined_mask.sum() > 0:\n                    candidates.append({\n                        "confidence": float(conf),\n                        "mask": refined_mask,\n                    })\n\n        # 2. Instance arbitration (trim)\n        final_instances = arbitrate_instances(\n            candidates,\n            disk_mask=disk_mask,\n            min_area=min_area,\n            overlap_mode="trim",\n        )\n\n        # Mandatory Host Sanitizer: zero shared pixels guaranteed & asserted\n        final_instances = sanitize_instances_zero_overlap(\n            final_instances,\n            min_area=min_area,\n        )\n\n        # 3. Pycocotools RLE encoding: ONE ROW PER ACTUAL DETECTED FILAMENT\n        if len(final_instances) == 0:\n            empty_disk_count += 1\n            rows_per_disk.append(0)\n            # Emit ZERO rows for images with zero detected filaments (per official host PQ metric contract)\n        else:\n            for k, inst in enumerate(final_instances):\n                mask = inst["mask"]\n                assert mask.sum() > 0, f"FATAL: Found zero-area predicted instance on {stem}_{k + 1}"\n                rle_str = encode_mask(mask)\n                all_rows.append({"filament_id": f"{stem}_{k + 1}", "segmentation_rle": rle_str})\n            rows_per_disk.append(len(final_instances))\n\n        if (idx + 1) % 25 == 0 or (idx + 1) == len(test_jpegs):\n            elapsed = time.perf_counter() - t0\n            print(f"  [{idx + 1:03d}/{len(test_jpegs):03d}] Processed {stem} | Total rows: {len(all_rows)} ({elapsed:.1f}s)")\n\n    total_time = time.perf_counter() - t0\n\n    # Safety assertion: model must produce positive predictions\n    assert len(all_rows) > 0, "FATAL: Submission is completely empty! No filaments predicted."\n\n    # Write submission CSV\n    out_path = Path(args.out)\n    out_path.parent.mkdir(parents=True, exist_ok=True)\n    with open(out_path, "w", newline="", encoding="utf-8") as f:\n        writer = csv.DictWriter(f, fieldnames=["filament_id", "segmentation_rle"])\n        writer.writeheader()\n        writer.writerows(all_rows)\n\n    # Statistics breakdown\n    r_arr = np.array(rows_per_disk)\n    covered_stems = sum(1 for r in rows_per_disk if r > 0)\n    empty_stems = sum(1 for r in rows_per_disk if r == 0)\n    mean_r = float(np.mean(r_arr))\n    p50_r = float(np.median(r_arr))\n    p90_r = float(np.percentile(r_arr, 90))\n\n    print("\\n" + "=" * 75)\n    print("SUBMISSION AUDIT & STATISTICS")\n    print("=" * 75)\n    print(f"  Submission CSV:           {out_path}")\n    print(f"  Total Test Images:        {len(test_jpegs)}")\n    print(f"  Covered Stems (>0 preds): {covered_stems} ({covered_stems / len(test_jpegs) * 100:.1f}%)")\n    print(f"  Empty Stems (0 preds):    {empty_stems} ({empty_stems / len(test_jpegs) * 100:.1f}%)")\n    print(f"  Total Filament Rows:      {len(all_rows)}")\n    print(f"  Rows per Image:           Mean={mean_r:.2f} | p50={p50_r:.1f} | p90={p90_r:.1f}")\n    print(f"  Total Elapsed Time:       {total_time:.2f} seconds ({len(test_jpegs) / total_time:.2f} img/s)")\n    print("=" * 75)\n\n    # Memory cleanup & VRAM logging\n    del yolo_model, refiner_model\n    if torch.cuda.is_available():\n        torch.cuda.empty_cache()\n        for dev_idx in range(torch.cuda.device_count()):\n            res_mb = torch.cuda.memory_reserved(dev_idx) / (1024 ** 2)\n            alloc_mb = torch.cuda.memory_allocated(dev_idx) / (1024 ** 2)\n            print(f"  [GPU {dev_idx}] Memory Reserved: {res_mb:.1f} MB | Allocated: {alloc_mb:.1f} MB")\n\n\ndef main():\n    parser = argparse.ArgumentParser(description="Run v8_1 Instance-Segmentation Cascade Inference")\n    parser.add_argument("--dry-run", action="store_true", help="Print frozen config and exit 0 without running")\n    parser.add_argument("--residual", type=int, default=0, choices=[0, 1], help="Enable residual discovery branch (default: 0)")\n    parser.add_argument("--conf", type=float, default=0.20, help="YOLO proposal confidence threshold (default: 0.20)")\n    parser.add_argument("--min-area", type=int, default=400, help="Minimum filament area in pixels (default: 400)")\n    parser.add_argument("--yolo-fallback", type=int, default=0, choices=[0, 1], help="Fallback to YOLO-seg mask if refiner output < min_area (default: 0)")\n    parser.add_argument("--overlap-mode", type=str, default="trim", choices=["trim", "allow"], help="Overlap arbitration: only \'trim\' permitted for submissions (default: trim)")\n    parser.add_argument("--tta", action="store_true", default=True, help="Use 4-flip TTA on crops")\n    parser.add_argument("--weights", type=str, default=None, help="Explicit path to YOLO weights (best.pt)")\n    parser.add_argument("--refiner-weights", type=str, default=None, help="Explicit path to Crop Refiner weights (crop_refiner_r34.pth)")\n    parser.add_argument("--out", type=str, default=str(CascadeConfig.SUBMISSION_PATH), help="Output submission CSV path")\n    args = parser.parse_args()\n\n    if args.dry_run:\n        print_infer_config(args)\n        print("Dry run completed successfully. Exiting 0 on CPU.\\n")\n        sys.exit(0)\n\n    if args.overlap_mode != "trim":\n        print(f"[FATAL ERROR] overlap_mode=\'{args.overlap_mode}\' is FORBIDDEN for submission generation!")\n        print("Host rejected submissions containing overlapping masks. Only overlap_mode=\'trim\' is permitted.")\n        sys.exit(1)\n\n    if args.residual == 1:\n        print("[NOTICE] Residual discovery branch is intentionally DISABLED for Baseline 1 (pure YOLO -> Crop Refiner).")\n\n    if not torch.cuda.is_available():\n        print("[ERROR] CUDA is not available! Cascade inference requires a GPU.")\n        print("Run with --dry-run to inspect configuration on CPU.")\n        sys.exit(1)\n\n    print_infer_config(args)\n    run_cascade_inference(args)\n\n\nif __name__ == "__main__":\n    main()\n')

with open("v8_1/4_sweep_inference.py", "w", encoding="utf-8") as f:
    f.write('"""\nv8_1 Stage 4 — Inference Diagnostic & Operating Point Grid Sweep (Directive R2).\n\nMaster: Grok | Executor: Antigravity\nTarget: Kaggle Solar Filament Segmentation Challenge 2026\n\nR2 Objectives:\n1. Stage A (Diagnose): Score validation physical JPEGs across each annotator observation separately.\n2. Stage B (Inference Sweep): Grid search over conf, min_area, yolo_mask_fallback, overlap_mode.\n   Default baseline: conf=0.25, min_area=100, yolo_mask_fallback=0, overlap_mode=trim (0.350 setting).\n   Select argmax pq_mean operating point and dump histogram.\n3. Stage C (Test Submit): Run inference on 180 test disks with frozen winner and verify contract.\n"""\n\nimport argparse\nimport csv\nimport json\nimport os\nimport sys\nimport time\nfrom collections import defaultdict\nfrom pathlib import Path\n\nimport cv2\nimport numpy as np\nimport torch\n\n# Ensure project root is on sys.path\nPROJECT_ROOT = Path(__file__).resolve().parent.parent\nsys.path.insert(0, str(PROJECT_ROOT))\n\nfrom metrics.pq import pq_score, encode_mask, decode_rle\nfrom v8_1.config import (\n    YOLOConfig,\n    CropRefinerConfig,\n    CascadeConfig,\n    MAGFILO_DIR,\n    YOLO_DATA_DIR,\n    OUT,\n    RUNS_DIR,\n    MODELS_DIR,\n)\nfrom v8_1.geometry import compute_adaptive_crop_side, square_bounds_from_bbox\nimport importlib\n_infer_module = importlib.import_module("v8_1.3_infer_cascade")\npreprocess_3ch = _infer_module.preprocess_3ch\nget_solar_disk_mask = _infer_module.get_solar_disk_mask\nrefine_crop_tta = _infer_module.refine_crop_tta\narbitrate_instances = _infer_module.arbitrate_instances\nrun_cascade_inference = _infer_module.run_cascade_inference\n\nif hasattr(sys.stdout, "reconfigure"):\n    sys.stdout.reconfigure(encoding="utf-8")\n\n\ndef load_validation_ground_truth(data_dir: Path, yolo_data_dir: Path):\n    """Load the 128 unique validation physical JPEGs and their ground truth annotator masks."""\n    ann_path = data_dir / "train" / "MAGFiLO_1.0_Annotations_kaggle2026_train.json"\n    if not ann_path.exists():\n        raise FileNotFoundError(f"MAGFiLO training JSON not found at: {ann_path}")\n\n    with open(ann_path, "r", encoding="utf-8") as f:\n        coco_data = json.load(f)\n\n    # Map image_id -> file_name and file_name -> list of image_ids\n    img_id_to_fn = {img["id"]: img["file_name"] for img in coco_data.get("images", [])}\n    fn_to_img_ids = defaultdict(list)\n    for iid, fn in img_id_to_fn.items():\n        fn_to_img_ids[fn].append(iid)\n\n    # Collect segmentation polygons per image_id (all categories 1-4 are filaments)\n    img_id_to_polys = defaultdict(list)\n    for ann in coco_data.get("annotations", []):\n        segs = ann.get("segmentation", [])\n        if isinstance(segs, list):\n            for poly in segs:\n                if len(poly) >= 6 and len(poly) % 2 == 0:\n                    img_id_to_polys[ann["image_id"]].append(poly)\n\n    # Identify val stems from yolo_data_dir labels\n    val_labels_dir = yolo_data_dir / "labels" / "val"\n    if val_labels_dir.exists():\n        val_label_files = list(val_labels_dir.glob("*.txt"))\n        val_physical_stems = sorted(list({p.stem.split("_ann_")[0] for p in val_label_files}))\n    else:\n        # Fallback to year GroupKFold logic if yolo_seg directory not yet materialized\n        all_stems = sorted(list(fn_to_img_ids.keys()))\n        years = [s[:4] for s in all_stems]\n        from sklearn.model_selection import GroupKFold\n        gkf = GroupKFold(n_splits=5)\n        splits = list(gkf.split(all_stems, groups=years))\n        _, val_idx = splits[0]\n        val_physical_stems = sorted([Path(all_stems[i]).stem for i in val_idx])\n\n    val_img_dir = data_dir / "train" / "train_images"\n    val_gt_by_stem = {}\n\n    for stem in val_physical_stems:\n        fn = f"{stem}.jpeg" if (val_img_dir / f"{stem}.jpeg").exists() else f"{stem}.jpg"\n        img_ids = fn_to_img_ids.get(fn, [])\n        annotator_polys = [img_id_to_polys.get(iid, []) for iid in img_ids]\n        val_gt_by_stem[stem] = {\n            "fn": fn,\n            "img_path": val_img_dir / fn,\n            "annotators_polys": annotator_polys,\n        }\n\n    return val_physical_stems, val_gt_by_stem\n\n\ndef extract_val_candidates_pool(\n    val_physical_stems: list[str],\n    val_gt_by_stem: dict,\n    yolo_model,\n    refiner_model,\n    device_yolo,\n    device_refiner,\n    base_conf: float = 0.10,\n) -> dict:\n    """Run YOLO proposer and Crop Refiner once at base_conf=0.10 on 128 val images.\n    \n    Caches candidates with RLE-compressed masks to eliminate redundant neural evaluations.\n    """\n    print(f"\\n[STAGE A] Extracting candidate pools on {len(val_physical_stems)} val disks (base_conf={base_conf:.2f})...")\n    candidates_by_stem = {}\n    t0 = time.perf_counter()\n\n    for idx, stem in enumerate(val_physical_stems):\n        img_path = val_gt_by_stem[stem]["img_path"]\n        raw = cv2.imread(str(img_path), cv2.IMREAD_GRAYSCALE)\n        if raw is None:\n            raise RuntimeError(f"FATAL: Unable to decode validation image at {img_path}")\n\n        h, w = raw.shape[:2]\n        img_3ch = preprocess_3ch(raw)\n\n        # Run YOLO at lowest sweep confidence\n        yolo_preds = yolo_model.predict(\n            source=str(img_path),\n            imgsz=YOLOConfig.IMGSZ,\n            conf=base_conf,\n            device=device_yolo,\n            verbose=False,\n        )\n\n        stem_candidates = []\n        if yolo_preds and yolo_preds[0].boxes is not None and len(yolo_preds[0].boxes) > 0:\n            boxes = yolo_preds[0].boxes.xyxy.cpu().numpy()\n            confs = yolo_preds[0].boxes.conf.cpu().numpy()\n            yolo_masks_raw = yolo_preds[0].masks.data.cpu().numpy() if (yolo_preds and yolo_preds[0].masks is not None) else None\n\n            for b_idx, (box, conf) in enumerate(zip(boxes, confs)):\n                x1 = max(0, int(box[0]))\n                y1 = max(0, int(box[1]))\n                x2 = min(w, int(box[2]))\n                y2 = min(h, int(box[3]))\n                bbox_2048 = [x1, y1, x2, y2]\n\n                if (x2 - x1) < 2 or (y2 - y1) < 2:\n                    continue\n\n                # Refine with Crop U-Net + 4-flip TTA\n                refined_mask = refine_crop_tta(\n                    refiner_model=refiner_model,\n                    img_3ch=img_3ch,\n                    bbox=bbox_2048,\n                    device=device_refiner,\n                    target_size=CropRefinerConfig.TARGET_SIZE,\n                    use_tta=True,\n                )\n\n                # Store YOLO-seg mask if available\n                yolo_m = None\n                if yolo_masks_raw is not None and b_idx < len(yolo_masks_raw):\n                    yolo_m = cv2.resize(yolo_masks_raw[b_idx].astype(np.uint8), (w, h), interpolation=cv2.INTER_NEAREST)\n\n                # Cache candidate as COCO RLE to keep RAM < 10 MB\n                stem_candidates.append({\n                    "confidence": float(conf),\n                    "bbox": bbox_2048,\n                    "refined_rle": encode_mask(refined_mask) if refined_mask.sum() > 0 else None,\n                    "refined_area": int(refined_mask.sum()),\n                    "yolo_rle": encode_mask(yolo_m) if (yolo_m is not None and yolo_m.sum() > 0) else None,\n                    "yolo_area": int(yolo_m.sum()) if yolo_m is not None else 0,\n                })\n\n        candidates_by_stem[stem] = stem_candidates\n\n        if (idx + 1) % 32 == 0 or (idx + 1) == len(val_physical_stems):\n            elapsed = time.perf_counter() - t0\n            print(f"  [{idx + 1:03d}/{len(val_physical_stems):03d}] Cached {stem} ({elapsed:.1f}s)")\n\n    print(f"[STAGE A] Extraction complete in {time.perf_counter() - t0:.1f}s.")\n    return candidates_by_stem\n\n\ndef evaluate_grid_point(\n    conf: float,\n    min_area: int,\n    yolo_fallback: int,\n    overlap_mode: str,\n    val_physical_stems: list[str],\n    val_gt_by_stem: dict,\n    candidates_by_stem: dict,\n    disk_mask: np.ndarray,\n) -> dict:\n    """Evaluate one parameter combination across all 128 validation disks."""\n    disk_pq_means, disk_pq_maxs = [], []\n    disk_sq_means, disk_rq_means = [], []\n    tot_tp, tot_fp, tot_fn = 0, 0, 0\n    tot_pred_instances = 0\n    preds_per_disk = []\n\n    for stem in val_physical_stems:\n        # 1. Rasterize ground-truth instances for all annotator observations on this physical image\n        annotators_polys = val_gt_by_stem[stem]["annotators_polys"]\n        annotator_instances = []\n        for polys in annotators_polys:\n            masks = []\n            for p in polys:\n                m = np.zeros((2048, 2048), dtype=np.uint8)\n                pts = np.asarray(p, dtype=np.int32).reshape(-1, 2)\n                cv2.fillPoly(m, [pts], 1)\n                if m.sum() > 0:\n                    masks.append(m)\n            annotator_instances.append(masks)\n\n        # 2. Select candidates for this operating point\n        raw_candidates = candidates_by_stem.get(stem, [])\n        filtered_candidates = []\n\n        for c in raw_candidates:\n            if c["confidence"] < conf:\n                continue\n\n            ref_area = c["refined_area"]\n            yolo_area = c["yolo_area"]\n\n            # YOLO mask fallback logic:\n            if yolo_fallback == 1 and ref_area < min_area and yolo_area >= min_area:\n                mask = decode_rle(c["yolo_rle"], 2048, 2048)\n            elif ref_area >= min_area and c["refined_rle"] is not None:\n                mask = decode_rle(c["refined_rle"], 2048, 2048)\n            else:\n                continue\n\n            filtered_candidates.append({\n                "confidence": c["confidence"],\n                "mask": mask,\n            })\n\n        # 3. Arbitrate instances using overlap_mode and min_area\n        accepted = arbitrate_instances(\n            filtered_candidates,\n            disk_mask=disk_mask,\n            min_area=min_area,\n            overlap_mode=overlap_mode,\n        )\n\n        pred_masks = [inst["mask"] for inst in accepted]\n        tot_pred_instances += len(pred_masks)\n        preds_per_disk.append(len(pred_masks))\n\n        # 4. Score against each annotator independently\n        ann_pqs, ann_sqs, ann_rqs = [], [], []\n        for ann_gt in annotator_instances:\n            score_dict = pq_score(pred_masks, ann_gt, iou_threshold=0.5)\n            ann_pqs.append(score_dict["PQ"])\n            ann_sqs.append(score_dict["SQ"])\n            ann_rqs.append(score_dict["RQ"])\n            tot_tp += score_dict["TP"]\n            tot_fp += score_dict["FP"]\n            tot_fn += score_dict["FN"]\n\n        if ann_pqs:\n            disk_pq_means.append(np.mean(ann_pqs))\n            disk_pq_maxs.append(np.max(ann_pqs))\n            disk_sq_means.append(np.mean(ann_sqs))\n            disk_rq_means.append(np.mean(ann_rqs))\n\n    mean_pq = float(np.mean(disk_pq_means)) if disk_pq_means else 0.0\n    max_pq = float(np.mean(disk_pq_maxs)) if disk_pq_maxs else 0.0\n    mean_sq = float(np.mean(disk_sq_means)) if disk_sq_means else 0.0\n    mean_rq = float(np.mean(disk_rq_means)) if disk_rq_means else 0.0\n\n    return {\n        "conf": conf,\n        "min_area": min_area,\n        "yolo_fallback": yolo_fallback,\n        "overlap_mode": overlap_mode,\n        "pq_mean": mean_pq,\n        "pq_max": max_pq,\n        "sq_mean": mean_sq,\n        "rq_mean": mean_rq,\n        "tp": tot_tp,\n        "fp": tot_fp,\n        "fn": tot_fn,\n        "n_pred": tot_pred_instances,\n        "preds_per_disk": preds_per_disk,\n    }\n\n\ndef run_r2_sweep(args):\n    """Execute complete Stage A, Stage B, and Stage C workflow."""\n    print("=" * 85)\n    print("V8_1 STAGE 4: GROK DIRECTIVE R2 INFERENCE & OPERATING POINT SWEEP")\n    print("=" * 85)\n\n    data_dir = MAGFILO_DIR\n    yolo_data_dir = YOLO_DATA_DIR\n    disk_mask = get_solar_disk_mask(2048, 2048, r_frac=CascadeConfig.SOLAR_DISK_R_FRAC)\n\n    # 1. Resolve weights\n    yolo_ckpt = YOLOConfig.resolve_weights(args.weights if hasattr(args, "weights") else None)\n    refiner_ckpt = CropRefinerConfig.resolve_ckpt(args.refiner_weights if hasattr(args, "refiner_weights") else None)\n\n    print(f"  YOLO Weights Resolved:    {yolo_ckpt} (exists: {yolo_ckpt.exists()})")\n    print(f"  Refiner Checkpoint:       {refiner_ckpt} (exists: {refiner_ckpt.exists()})")\n\n    if not yolo_ckpt.exists() or not refiner_ckpt.exists():\n        if args.check_weights_only:\n            print("\\n[CHECK-WEIGHTS-ONLY] Weights missing. Retraining required.")\n            return None\n        raise FileNotFoundError(\n            f"FATAL: Weights missing! YOLO: {yolo_ckpt} ({yolo_ckpt.exists()}), Refiner: {refiner_ckpt} ({refiner_ckpt.exists()}). "\n            f"Attach completed kernel output or run training cells first."\n        )\n\n    if args.check_weights_only:\n        print("\\n[CHECK-WEIGHTS-ONLY] Weights confirmed present! Skipping retraining.")\n        return True\n\n    # 2. Load Models\n    from ultralytics import YOLO\n    import segmentation_models_pytorch as smp\n\n    n_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 0\n    device_yolo = 0 if n_gpus > 0 else "cpu"\n    refiner_gpu_idx = 1 if n_gpus >= 2 else (0 if n_gpus == 1 else "cpu")\n    device_refiner = torch.device(f"cuda:{refiner_gpu_idx}" if n_gpus > 0 else "cpu")\n\n    print(f"  Model Placement: YOLO -> cuda:{device_yolo} | Refiner -> {device_refiner}")\n\n    yolo_model = YOLO(str(yolo_ckpt))\n    refiner_model = smp.Unet(\n        encoder_name=CropRefinerConfig.ENCODER,\n        in_channels=CropRefinerConfig.IN_CHANNELS,\n        classes=CropRefinerConfig.CLASSES,\n    ).to(device_refiner)\n    refiner_model.load_state_dict(torch.load(refiner_ckpt, map_location=device_refiner))\n    refiner_model.eval()\n\n    # 3. Load Validation Set & Annotations\n    val_physical_stems, val_gt_by_stem = load_validation_ground_truth(data_dir, yolo_data_dir)\n    print(f"  Holdout unique physical validation JPEGs: {len(val_physical_stems)}")\n\n    # Ground truth instances per image statistics\n    gt_counts_per_disk = []\n    for stem in val_physical_stems:\n        for polys in val_gt_by_stem[stem]["annotators_polys"]:\n            gt_counts_per_disk.append(len(polys))\n    gt_counts_arr = np.array(gt_counts_per_disk)\n    print(f"  GT Instances per Image (all annotators): Mean={gt_counts_arr.mean():.2f} | Median={np.median(gt_counts_arr):.1f} | Min={gt_counts_arr.min()} | Max={gt_counts_arr.max()}")\n\n    # 4. Stage A: Extract candidate pools (runs heavy neural forward passes once)\n    candidates_by_stem = extract_val_candidates_pool(\n        val_physical_stems,\n        val_gt_by_stem,\n        yolo_model,\n        refiner_model,\n        device_yolo,\n        device_refiner,\n        base_conf=0.10,\n    )\n\n    # 5. Stage B: Inference Grid Sweep\n    confs = [0.10, 0.15, 0.20, 0.25, 0.35]\n    min_areas = [50, 100, 200, 400]\n    fallbacks = [0, 1]\n    overlap_modes = ["trim", "allow"]\n\n    total_points = len(confs) * len(min_areas) * len(fallbacks) * len(overlap_modes)\n    print(f"\\n[STAGE B] Running Inference Grid Search across {total_points} operating points...")\n\n    results_table = []\n    t_sweep_start = time.perf_counter()\n\n    for conf in confs:\n        for ma in min_areas:\n            for fb in fallbacks:\n                for om in overlap_modes:\n                    res = evaluate_grid_point(\n                        conf=conf,\n                        min_area=ma,\n                        yolo_fallback=fb,\n                        overlap_mode=om,\n                        val_physical_stems=val_physical_stems,\n                        val_gt_by_stem=val_gt_by_stem,\n                        candidates_by_stem=candidates_by_stem,\n                        disk_mask=disk_mask,\n                    )\n                    results_table.append(res)\n\n    print(f"[STAGE B] Sweep completed in {time.perf_counter() - t_sweep_start:.2f}s.\\n")\n\n    # 6. Format and Print Required VAL_PQ_TABLE\n    print("=" * 115)\n    print("## VAL_PQ_TABLE")\n    print("=" * 115)\n    header = f"{\'conf\':<6} | {\'min_area\':<8} | {\'fallback\':<8} | {\'overlap\':<7} | {\'PQ_mean\':<8} | {\'PQ_max\':<8} | {\'SQ\':<7} | {\'RQ\':<7} | {\'TP\':<6} | {\'FP\':<6} | {\'FN\':<6} | {\'n_pred\':<6}"\n    print(header)\n    print("-" * 115)\n\n    baseline_result = None\n    for r in results_table:\n        is_baseline = (r["conf"] == 0.25 and r["min_area"] == 100 and r["yolo_fallback"] == 0 and r["overlap_mode"] == "trim")\n        if is_baseline:\n            baseline_result = r\n        marker = " <-- [0.350 BASELINE]" if is_baseline else ""\n        row_str = (\n            f"{r[\'conf\']:<6.2f} | {r[\'min_area\']:<8} | {r[\'yolo_fallback\']:<8} | {r[\'overlap_mode\']:<7} | "\n            f"{r[\'pq_mean\']:<8.4f} | {r[\'pq_max\']:<8.4f} | {r[\'sq_mean\']:<7.4f} | {r[\'rq_mean\']:<7.4f} | "\n            f"{r[\'tp\']:<6} | {r[\'fp\']:<6} | {r[\'fn\']:<6} | {r[\'n_pred\']:<6}{marker}"\n        )\n        print(row_str)\n    print("=" * 115)\n\n    # 7. Select Argmax Operating Point\n    best_config = max(results_table, key=lambda x: x["pq_mean"])\n    print("\\n## CHOSEN_OPERATING_POINT")\n    print(f"  Optimal Configuration by pq_mean:")\n    print(f"    conf:                {best_config[\'conf\']}")\n    print(f"    min_area:            {best_config[\'min_area\']}")\n    print(f"    yolo_mask_fallback:  {best_config[\'yolo_fallback\']}")\n    print(f"    overlap_mode:        {best_config[\'overlap_mode\']}")\n    print(f"    Validation PQ_mean:  {best_config[\'pq_mean\']:.4f} (Baseline 0.350: {baseline_result[\'pq_mean\']:.4f})")\n    print(f"    Validation PQ_max:   {best_config[\'pq_max\']:.4f}")\n    print(f"    Validation SQ:       {best_config[\'sq_mean\']:.4f}")\n    print(f"    Validation RQ:       {best_config[\'rq_mean\']:.4f}")\n    print(f"    TP / FP / FN:        {best_config[\'tp\']} / {best_config[\'fp\']} / {best_config[\'fn\']}")\n    print(f"    Total Pred Filaments:{best_config[\'n_pred\']}")\n\n    # 8. Histogram: Predictions per Image vs Ground Truth Instances\n    pred_counts_arr = np.array(best_config["preds_per_disk"])\n    print("\\n--- INSTANCE HISTOGRAM COMPARISON (Fold-0 Validation) ---")\n    print(f"  Ground Truth / Disk: Mean={gt_counts_arr.mean():.2f} | Median={np.median(gt_counts_arr):.1f} | Min={gt_counts_arr.min()} | Max={gt_counts_arr.max()}")\n    print(f"  Chosen Preds / Disk: Mean={pred_counts_arr.mean():.2f} | Median={np.median(pred_counts_arr):.1f} | Min={pred_counts_arr.min()} | Max={pred_counts_arr.max()}")\n\n    # Save summary JSON\n    summary_path = OUT / "v8_1_r2_sweep_summary.json"\n    with open(summary_path, "w", encoding="utf-8") as f:\n        json.dump({\n            "baseline": {k: v for k, v in baseline_result.items() if k != "preds_per_disk"},\n            "chosen_operating_point": {k: v for k, v in best_config.items() if k != "preds_per_disk"},\n            "all_results": [{k: v for k, v in r.items() if k != "preds_per_disk"} for r in results_table],\n        }, f, indent=2)\n    print(f"Saved sweep summary to {summary_path}")\n\n    # 9. Stage C: Test Submission Generation\n    if args.run_test:\n        print("\\n" + "=" * 85)\n        print("[STAGE C] Generating Test Submission using Frozen Operating Point...")\n        print("=" * 85)\n        \n        # Prepare namespace for run_cascade_inference\n        test_args = argparse.Namespace(\n            conf=best_config["conf"],\n            min_area=best_config["min_area"],\n            yolo_fallback=best_config["yolo_fallback"],\n            overlap_mode=best_config["overlap_mode"],\n            tta=True,\n            weights=str(yolo_ckpt),\n            refiner_weights=str(refiner_ckpt),\n            out=args.out,\n        )\n        run_cascade_inference(test_args)\n\n    return best_config\n\n\ndef main():\n    parser = argparse.ArgumentParser(description="v8_1 R2 Diagnostic & Inference Sweep")\n    parser.add_argument("--dry-run", action="store_true", help="Print config and exit 0")\n    parser.add_argument("--check-weights-only", action="store_true", help="Check if weights exist without running sweep")\n    parser.add_argument("--weights", type=str, default=None, help="Explicit path to YOLO weights (best.pt)")\n    parser.add_argument("--refiner-weights", type=str, default=None, help="Explicit path to Refiner weights (crop_refiner_r34.pth)")\n    parser.add_argument("--run-test", action="store_true", default=True, help="Execute Stage C test inference with winning operating point")\n    parser.add_argument("--out", type=str, default=str(OUT / "submission.csv"), help="Output submission CSV path")\n    args = parser.parse_args()\n\n    if args.dry_run:\n        print("v8_1 4_sweep_inference.py dry run OK. Exiting 0 on CPU.")\n        sys.exit(0)\n\n    if not torch.cuda.is_available() and not args.check_weights_only:\n        print("[ERROR] CUDA is not available! R2 inference sweep requires GPU.")\n        sys.exit(1)\n\n    run_r2_sweep(args)\n\n\nif __name__ == "__main__":\n    main()\n')

print("✅ Successfully deployed metrics/, scripts/, and v8_1/ modules into working directory.")


In [ ]:
# ==============================================================================
# CELL 5: Stage 0 — Dataset Conversion & Split Verification
# Purpose: Convert MAGFiLO COCO annotations to YOLO-seg polygon format.
#          - Enforces GroupKFold by year prefix (0 physical filename leakage)
#          - Produces 579 train images and 128 validation images
#          - Verifies data.yaml and generates conversion_report.json
# Runtime: ~1-2 minutes
# ==============================================================================
import time
t0_stage0 = time.perf_counter()
from pathlib import Path
import subprocess, sys

out = Path("/kaggle/working/data/yolo_seg")
cmd = [sys.executable, "scripts/convert_coco_to_yolo.py",
       "--data-dir", str(data_dir),
       "--out-dir", str(out)]
print("CONVERT:", cmd, flush=True)
subprocess.check_call(cmd)

# Verify conversion results
yaml_path = out / "data.yaml"
assert yaml_path.exists(), f"FATAL: data.yaml not found at {yaml_path}!"
print("\n--- data.yaml content ---")
with open(yaml_path) as f:
    print(f.read())

n_trn = len(list((out / "images" / "train").glob("*.jpeg")))
n_val = len(list((out / "images" / "val").glob("*.jpeg")))
print(f"Verified YOLO dataset images: {n_trn} train images, {n_val} val images")
assert n_trn > 0 and n_val > 0, f"FATAL: Empty dataset! train={n_trn}, val={n_val}"

# Report annotator-level samples vs unique physical JPEGs
import json
report_path = out / "conversion_report.json"
if report_path.exists():
    with open(report_path, "r", encoding="utf-8") as rf:
        rep = json.load(rf)
    print(f"Annotator-level samples: {rep.get('train_samples_count')} train, {rep.get('val_samples_count')} val")
    print(f"Unique physical JPEGs:   {rep.get('train_unique_jpegs')} train, {rep.get('val_unique_jpegs')} val")

t_stage0 = time.perf_counter() - t0_stage0
print(f"\n[TIMING] Stage 0 (Dataset Conversion): {t_stage0:.2f}s ({t_stage0 / 60:.2f} min)")


In [ ]:
# ==============================================================================
# CELL 6: Stage 1 — YOLO11s-seg Proposer Training & Weights Resolution
# Purpose: Check if pretrained weights exist. If present, skip training!
#          If missing, train YOLO11s-seg (batch 2, 20 epochs, cuda:0).
# Runtime: ~10 seconds if weights present, ~20-25 minutes if training
# ==============================================================================
import time
t0_stage1 = time.perf_counter()
from pathlib import Path
from v8_1.config import YOLOConfig

existing_yolo = YOLOConfig.resolve_weights()
if existing_yolo.exists() and "weights/best.pt" in str(existing_yolo):
    print(f"✅ Existing Stage 1 YOLO weights detected at: {existing_yolo}")
    print("   Skipping Stage 1 training to preserve compute for R2 inference sweep.")
else:
    print("[INFO] Stage 1 YOLO weights not found. Launching training from scratch...")
    !python v8_1/1_train_yolo.py --batch 2 --epochs 20

t_stage1 = time.perf_counter() - t0_stage1
print(f"\n[TIMING] Stage 1: {t_stage1:.2f}s ({t_stage1 / 60:.2f} min)")


In [ ]:
# ==============================================================================
# CELL 7: Stage 2 — Crop U-Net Refiner Training & Weights Resolution
# Purpose: Check if pretrained refiner weights exist. If present, skip training!
#          If missing, train Crop Refiner U-Net (batch 8, 20 epochs, cuda:0).
# Runtime: ~10 seconds if weights present, ~15-20 minutes if training
# ==============================================================================
import time
t0_stage2 = time.perf_counter()
from pathlib import Path
from v8_1.config import CropRefinerConfig

existing_refiner = CropRefinerConfig.resolve_ckpt()
if existing_refiner.exists() and "crop_refiner_r34.pth" in str(existing_refiner):
    print(f"✅ Existing Stage 2 Refiner checkpoint detected at: {existing_refiner}")
    print("   Skipping Stage 2 training to preserve compute for R2 inference sweep.")
else:
    print("[INFO] Stage 2 Refiner weights not found. Launching training from scratch...")
    !python v8_1/2_train_crop_refiner.py --boxes-from-gt

t_stage2 = time.perf_counter() - t0_stage2
print(f"\n[TIMING] Stage 2: {t_stage2:.2f}s ({t_stage2 / 60:.2f} min)")


In [ ]:
# ==============================================================================
# CELL 8: Stage 3 — Grok Directive R3 Host-Safe Trim Test Submission
# Purpose: Generate test submission using frozen R3 knobs:
#          - conf: 0.20
#          - min_area: 400
#          - yolo_fallback: 0
#          - overlap_mode: trim (host-safe, zero shared pixels)
#          - tta: True (4-flip TTA on crops)
#          - Weights: best.pt and crop_refiner_r34.pth from notebooka464bcfe13
# Hardware: Dual T4 Model Parallelism (YOLO on cuda:0, Refiner on cuda:1)
# Runtime: ~3-5 minutes (Stage C only — no retraining, no sweep)
# Output Submission: /kaggle/working/submission.csv
# ==============================================================================
import time
t0_stage3 = time.perf_counter()

from pathlib import Path
from v8_1.config import YOLOConfig, CropRefinerConfig

best_yolo = YOLOConfig.resolve_weights()
best_refiner = CropRefinerConfig.resolve_ckpt()

print(f"R3 Proposer Weights: {best_yolo}")
print(f"R3 Refiner Weights:  {best_refiner}")
assert best_yolo.exists(), f"FATAL: YOLO weights missing at {best_yolo}!"
assert best_refiner.exists(), f"FATAL: Refiner weights missing at {best_refiner}!"

!python v8_1/3_infer_cascade.py \
    --conf 0.20 \
    --min-area 400 \
    --yolo-fallback 0 \
    --overlap-mode trim \
    --tta \
    --weights "{best_yolo}" \
    --refiner-weights "{best_refiner}" \
    --out /kaggle/working/submission.csv

t_stage3 = time.perf_counter() - t0_stage3
print(f"\n[TIMING] Stage 3 R3 Host-Safe Trim Inference: {t_stage3:.2f}s ({t_stage3 / 60:.2f} min)")


In [ ]:
# ==============================================================================
# CELL 9: Stage 4 — Final Kaggle Submission Audit & Host Contract Verification
# Purpose: Rigorously audit the generated submission against the official host contract:
#          - Verifies columns: filament_id, segmentation_rle
#          - Verifies all 180 official test image stems are accounted for
#          - Confirms no duplicate filament_id values
#          - Decodes every RLE to ensure shape == (2048, 2048) and sum > 0 (strictly positive area)
#          - MANDATORY HOST CHECK: Asserts ZERO pairwise mask overlap per disk:
#            sum(mask_i & mask_j) == 0 for all i != j
#          - Confirms disks with 0 detections emit 0 rows (no penalized dummy masks)
#          - Prints summary statistics and total notebook wall-clock time
# Runtime: ~30 seconds
# ==============================================================================
import time
t0_stage4 = time.perf_counter()

import pandas as pd
import numpy as np
from collections import defaultdict
from pathlib import Path
from metrics.pq import decode_rle

sub_path = Path("/kaggle/working/submission.csv")
assert sub_path.exists(), "FATAL: submission.csv was not generated!"

df = pd.read_csv(sub_path)
print("=" * 75)
print("FINAL KAGGLE SUBMISSION AUDIT & HOST CONTRACT VERIFICATION")
print("=" * 75)
print(f"File Path:        {sub_path}")
print(f"Columns:          {list(df.columns)}")
assert list(df.columns) == ["filament_id", "segmentation_rle"], "FATAL: Invalid columns!"
assert len(df) > 0, "FATAL: Submission file has 0 rows!"
assert df["filament_id"].is_unique, "FATAL: Duplicate filament_id found!"

# Verify all stems belong to official test set (180 images)
test_dir = data_dir / "test" / "test_images"
official_test_stems = {p.stem for p in (list(test_dir.glob("*.jpeg")) + list(test_dir.glob("*.jpg")))}
print(f"Official Test JPEGs on disk: {len(official_test_stems)}")
assert len(official_test_stems) == 180, f"Expected 180 official test JPEGs, found {len(official_test_stems)}"

stems = df["filament_id"].apply(lambda x: x.rsplit("_", 1)[0])
unknown_stems = set(stems) - official_test_stems
assert len(unknown_stems) == 0, f"FATAL: Unknown test stems in submission: {unknown_stems}"

covered_stems = set(stems)
missing_prediction_stems = official_test_stems - covered_stems

print(f"Total Predicted Rows:     {len(df)}")
print(f"Covered Test Disks:       {len(covered_stems)} / 180 ({len(covered_stems) / 180 * 100:.1f}%)")
print(f"Zero-Prediction Disks:    {len(missing_prediction_stems)} / 180 ({len(missing_prediction_stems) / 180 * 100:.1f}%)")

# Group predictions by disk stem to decode and audit zero-overlap
stem_to_rles = defaultdict(list)
for idx, row in df.iterrows():
    s = row["filament_id"].rsplit("_", 1)[0]
    stem_to_rles[s].append((row["filament_id"], row["segmentation_rle"]))

# Verify every submitted RLE decodes to (2048, 2048) with strictly positive area (> 0)
# and MANDATORY PAIRWISE OVERLAP ASSERTION per disk
print("\nVerifying RLE decode, area bounds, and ZERO pairwise overlap per disk...")
total_pairwise_checks = 0
for stem, items in stem_to_rles.items():
    masks = []
    for fid, rle in items:
        m = decode_rle(rle, 2048, 2048)
        assert m.shape == (2048, 2048), f"Row {fid} invalid decoded shape: {m.shape}"
        assert m.sum() > 0, f"FATAL: Row {fid} has 0 active pixels! No dummy zero masks permitted."
        masks.append(m)
    
    # Check pairwise overlap on this disk
    n_m = len(masks)
    if n_m > 1:
        occupied = np.zeros((2048, 2048), dtype=np.uint8)
        for k in range(n_m):
            occupied = np.maximum(occupied, masks[k])
        
        for i in range(n_m):
            for j in range(i + 1, n_m):
                overlap = int(np.logical_and(masks[i], masks[j]).sum())
                assert overlap == 0, (
                    f"FATAL: Host overlap violation on disk {stem}! "
                    f"Instances {items[i][0]} and {items[j][0]} share {overlap} pixels! "
                    f"Submissions may not contain overlapping masks."
                )
                total_pairwise_checks += 1
        
        assert sum(int(m.sum()) for m in masks) == int(occupied.sum()), (
            f"FATAL: Disjoint sum check failed on {stem}!"
        )

print(f"✅ OVERLAP ASSERTION PASS: Verified zero shared pixels across all test disks ({total_pairwise_checks} pairwise checks passed)!")

counts_per_disk = stems.value_counts()
print(f"Instances / Covered Disk: Mean={counts_per_disk.mean():.2f} | p50={counts_per_disk.median():.1f} | Min={counts_per_disk.min()} | Max={counts_per_disk.max()}")

print("\nFirst 10 Rows:")
print(df.head(10))

t_stage4 = time.perf_counter() - t0_stage4
t_total_notebook = time.perf_counter() - t_notebook_start

print("=" * 75)
print(f"[TIMING] Stage 4 Audit:          {t_stage4:.2f}s ({t_stage4 / 60:.2f} min)")
print(f"[TIMING] Total Notebook Time:    {t_total_notebook:.2f}s ({t_total_notebook / 60:.2f} min)")
print("=" * 75)
print("[OK] ALL HOST SUBMISSION CONTRACT CHECKS PASSED (ZERO OVERLAP VERIFIED)!")
